In [1]:
# Tutorial de datos con Python: pandas + NumPy + MongoDB
# Generado para: Andrés Hormazábal — UDP
# Material de apoyo del sistema de 25 notebooks (acompaña a NB-02)
# Fecha de generación: 2026-10-30

# Tutorial de datos: pandas + NumPy + MongoDB

**Bloque:** 1 — Fundamentos de Datos y Cómputo · *material de apoyo de NB-02*

**Base bibliográfica:** VanderPlas, *Python Data Science Handbook* (cap. 13–24) ·
McKinney, *Python for Data Analysis* (cap. 5–11) · Kleppmann, *DDIA* (cap. 2–3:
modelo documental; cap. 5–6: replicación y particionamiento) · documentación oficial
de MongoDB y PyMongo

**Prerequisitos:** Python básico. NB-01 ayuda pero no es obligatorio: aquí se explica
lo necesario de NumPy sobre la marcha.

## Qué es esto y en qué se diferencia de la guía

| | `guia_pandas` | **este tutorial** | NB-02 |
|---|---|---|---|
| Forma | catálogo de referencia | **recorrido completo** | caso de estudio de punta a punta |
| Para | buscar una función | **aprender el flujo** | ver el flujo aplicado con decisiones |
| Incluye MongoDB | solo la salida | **sí, Parte F completa** | sí, con índices y agregación |

## Objetivos de Aprendizaje

Al completar este tutorial serás capaz de:

1. **Construir** un pipeline ETL completo: extraer → perfilar → limpiar → transformar →
   validar → cargar, con cada paso aislado en su función.
2. **Diagnosticar** un dataset desconocido y decidir qué hacer con nulos, duplicados y
   tipos mezclados, justificando cada decisión.
3. **Implementar** medidas de similitud (Jaccard, Jaccard ponderado, coseno) y entender
   por qué el IDF cambia los resultados.
4. **Operar** MongoDB desde Python: CRUD, consultas sobre arrays, agregación, `$lookup`.
5. **Medir** el efecto de un índice con `explain()` y leer un plan de ejecución.
6. **Evitar** los errores que no lanzan excepción: fuga de datos, merges que multiplican,
   `NaN` insertados en Mongo.

| Curso / Área | Relevancia |
|---|---|
| **Big Data (CIT-2309)** | **Referencia directa del proyecto**: ETL con pandas + almacenamiento en MongoDB + análisis con aggregation pipeline. La Parte F es la Entrega 2. |
| IA (CIT-2013) | Los datos limpios y bien tipados son el 80 % del trabajo antes de entrenar (NB-09, NB-11). |
| Sistemas Distribuidos (CIT-2011) | §29 (réplicas y sharding) conecta con replicación líder-seguidor y particionamiento de Kleppmann. |
| Comunicaciones Digitales | Series de mediciones con `.dt`, ventanas móviles y acumulados (§18). |
| Tesis SNN/Ojo | Registrar experimentos (parámetros + métricas de ojo + BER) como documentos de esquema flexible. |
| Notebooks relacionados | NB-01 (base), **NB-02** (caso completo), NB-03 (vectorización), NB-22 (este ETL como DAG de Airflow) |

<a id="indice"></a>
## Índice

Haz clic en una sección para saltar a ella; cada sección tiene un enlace *↑ índice* para volver aquí.

- [Configuración](#config)
- **[Parte A · Fundamentos](#parte-a)**
  [1. Estructuras](#sec-1) · [2. Leer y escribir archivos](#sec-2) · [3. Perfilar](#sec-3) · [4. Seleccionar columnas y filas](#sec-4) · [5. Filtrar con condiciones](#sec-5)
- **[Parte B · Limpieza](#parte-b)**
  [6. Valores nulos](#sec-6) · [7. Tipos de datos](#sec-7) · [8. Duplicados](#sec-8) · [9. Texto con .str](#sec-9) · [10. Fechas y horas](#sec-10) · [11. Aplicar funciones](#sec-11)
- **[Parte C · Transformar](#parte-c)**
  [12. Columnas con varios valores](#sec-12) · [13. Mapeo con diccionarios y tablas de equivalencia](#sec-13) · [14. Agrupar y agregar](#sec-14) · [15. Unir tablas](#sec-15) · [16. Reestructurar](#sec-16) · [17. Ordenar, ranking y top-N](#sec-17) · [18. Ventanas y acumulados](#sec-18)
- **[Parte D · Matemática y similitud](#parte-d)**
  [19. NumPy](#sec-19) · [20. Datos aleatorios y sintéticos](#sec-20) · [21. Similitud](#sec-21)
- **[Parte E · Calidad y rendimiento](#parte-e)**
  [22. Validación](#sec-22) · [23. Rendimiento](#sec-23)
- **[Parte F · MongoDB](#parte-f)**
  [24. Conceptos](#sec-24) · [25. De DataFrame a documentos](#sec-25) · [26. CRUD](#sec-26) · [27. Consultas y agregación](#sec-27) · [28. Índices y explain()](#sec-28) · [29. Réplicas y particionamiento](#sec-29)
- **[Parte G · Cierre](#parte-g)**
  [30. Plantilla de ETL completo](#sec-30) · [31. 🔧 Pruébalo tú mismo](#sec-31) · [32. Cheat Sheet](#sec-32) · [33. Conexiones con el Mundo Real](#sec-33) · [34. Ejercicios](#sec-34) · [35. Glosario](#sec-35) · [36. Conclusión y verificación](#sec-36)


<a id="config"></a>
## Configuración

> **NOTA sobre cómo leer este tutorial.** La versión anterior apilaba hasta quince
> expresiones por celda, y Jupyter **solo muestra la última**. Aquí cada línea deja su
> resultado visible con el helper `ver()`, para que puedas leerlo sin ejecutarlo.
>
> **NOTA pandas 3.x.** El texto ahora usa dtype `str` (antes `object`) y
> *Copy-on-Write* está activo por defecto: las asignaciones encadenadas del tipo
> `df[df.a > 1]["b"] = 0` ya **no** modifican el original (antes a veces sí, a veces no).
> Si vienes de pandas 1.x, ese cambio te va a ahorrar errores.

In [2]:
import ast                    # convertir texto que "parece" lista/dict en objeto real
import json                   # serializar a JSON (para verificar documentos)
import math                   # sqrt, isnan — operaciones escalares
import re                     # expresiones regulares
import tempfile               # carpeta temporal para escribir sin ensuciar el proyecto
import time                   # medir tiempos de ejecución
import unicodedata            # quitar tildes correctamente
from collections import Counter   # contador para acumular por pedazos
from io import StringIO           # simular archivos en memoria
from pathlib import Path          # rutas portables entre sistemas operativos

import numpy as np
import pandas as pd
from IPython.display import display

print("pandas", pd.__version__, "| numpy", np.__version__)

pd.set_option("display.max_columns", 30)     # no cortar en 10 columnas
pd.set_option("display.max_colwidth", 60)    # ancho máximo por celda de texto
pd.set_option("display.width", 160)          # ancho de consola antes de partir líneas


def ver(etiqueta: str, obj, maxfilas: int = 8) -> None:
    """Muestra `obj` con una etiqueta.

    Existe porque Jupyter solo muestra la ÚLTIMA expresión de cada celda: sin
    esto, en una celda-catálogo de 12 líneas se perderían 11 resultados.
    """
    print(f"\n▸ {etiqueta}")
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        display(obj.head(maxfilas) if len(obj) > maxfilas else obj)
    else:
        print(obj)


TMP = Path(tempfile.mkdtemp(prefix="tutorial_"))   # carpeta temporal para los ejemplos
print("Carpeta temporal de trabajo:", TMP)

pandas 3.0.6 | numpy 2.5.3
Carpeta temporal de trabajo: /var/folders/_t/ttrqn_wj631b9p3_5t5hwbzh0000gn/T/tutorial_fcsipd77


---

<a id="parte-a"></a>
# Parte A · Fundamentos

<a id="sec-1"></a>
## 1. Estructuras: `Series` y `DataFrame`
<sub>[↑ índice](#indice)</sub>

El modelo mental correcto, y el que evita la mitad de los errores:

```
Series    = un array de NumPy  +  un ÍNDICE con etiquetas
DataFrame = un diccionario de Series que COMPARTEN el mismo índice

        índice      columna "nombre"   columna "puntaje"
          0      →      "Ana"      |      7.5
          1      →      "Luis"     |      8.0     ← cada columna es una Series
          2      →      "Eva"      |      NaN
```

La consecuencia práctica: **el índice se alinea solo**. Cuando sumas dos Series, pandas
empareja por etiqueta, no por posición. Eso es lo que hace que una operación sobre
columnas de distinta longitud produzca `NaN` en vez de un error.

In [3]:
# --- Series: un vector con etiquetas ---
s = pd.Series([10, 20, 30],              # los datos
              index=["a", "b", "c"],     # las etiquetas (si se omite: 0,1,2...)
              name="valor")              # nombre de la Series (será el de la columna)
ver("La Series completa", s)
ver("Acceso por ETIQUETA s['b']", s["b"])
ver("Acceso por POSICIÓN s.iloc[0]", s.iloc[0])
ver("Los datos crudos (array de NumPy)", s.values)
ver("Las etiquetas", s.index.tolist())

# --- DataFrame desde un dict de columnas: la forma más común ---
df = pd.DataFrame({
    "id":      [1, 2, 3, 4],
    "nombre":  ["Ana", "Luis", "Eva", "Raúl"],
    "puntaje": [7.5, 8.0, np.nan, 6.1],    # el NaN obliga a que la columna sea float
})
ver("DataFrame desde dict de columnas", df)

# --- Desde lista de dicts: así llegan los JSON y los documentos de Mongo ---
df2 = pd.DataFrame([{"id": 1, "x": "a"},
                    {"id": 2, "x": "b", "extra": True}])   # la clave que falta -> NaN
ver("Desde lista de dicts (claves faltantes -> NaN)", df2)

# --- Desde lista de listas: hay que nombrar las columnas a mano ---
df3 = pd.DataFrame([[1, "a"], [2, "b"]], columns=["id", "x"])
ver("Desde lista de listas", df3)

# La alineación automática por índice, que es la propiedad clave de pandas
izq = pd.Series([1, 2, 3], index=["a", "b", "c"])
der = pd.Series([10, 20], index=["b", "c"])
ver("Suma de Series con índices DISTINTOS", izq + der)
print("""
  Fíjate: 'a' no existe en la segunda Series, así que el resultado es NaN en vez
  de un error. pandas alinea por ETIQUETA, no por posición. Esto es una ventaja
  enorme cuando las tablas vienen de fuentes distintas, y una fuente de sorpresas
  cuando esperabas una suma posicional.""")


▸ La Series completa


a    10
b    20
c    30
Name: valor, dtype: int64


▸ Acceso por ETIQUETA s['b']
20

▸ Acceso por POSICIÓN s.iloc[0]
10

▸ Los datos crudos (array de NumPy)
[10 20 30]

▸ Las etiquetas
['a', 'b', 'c']

▸ DataFrame desde dict de columnas


,id,nombre,puntaje
0,1,Ana,7.5
1,2,Luis,8.0
2,3,Eva,NaN
3,4,Raúl,6.1



▸ Desde lista de dicts (claves faltantes -> NaN)


...


▸ Desde lista de listas


,id,x
0,1,a
1,2,b



▸ Suma de Series con índices DISTINTOS


a     NaN
b    12.0
c    23.0
dtype: float64


  Fíjate: 'a' no existe en la segunda Series, así que el resultado es NaN en vez
  de un error. pandas alinea por ETIQUETA, no por posición. Esto es una ventaja
  enorme cuando las tablas vienen de fuentes distintas, y una fuente de sorpresas
  cuando esperabas una suma posicional.


In [4]:
ver("shape (filas, columnas)", df.shape)
ver("ndim: número de dimensiones (2 = tabla)", df.ndim)
ver("size: total de celdas (filas × columnas)", df.size)
ver("columns: nombres de columna", df.columns.tolist())
ver("index: las etiquetas de fila", df.index)
ver("dtypes: tipo de cada columna", df.dtypes)
ver("len(df): número de FILAS (no de columnas)", len(df))
ver("T: transpuesta (filas <-> columnas)", df.T)
ver("to_numpy(): salir de pandas hacia NumPy", df.to_numpy())
print("""
  ⚠️ to_numpy() sobre columnas de tipos MEZCLADOS devuelve un array de object,
  que pierde la ventaja de velocidad de NumPy. Si vas a hacer cálculo numérico,
  selecciona primero las columnas numéricas con select_dtypes('number').""")


▸ shape (filas, columnas)
(4, 3)

▸ ndim: número de dimensiones (2 = tabla)
2

▸ size: total de celdas (filas × columnas)
12

▸ columns: nombres de columna
['id', 'nombre', 'puntaje']

▸ index: las etiquetas de fila
RangeIndex(start=0, stop=4, step=1)

▸ dtypes: tipo de cada columna


id           int64
nombre         str
puntaje    float64
dtype: object


▸ len(df): número de FILAS (no de columnas)
4

▸ T: transpuesta (filas <-> columnas)


,0,1,2,3
id,1,2,3,4
nombre,Ana,Luis,Eva,Raúl
puntaje,7.5,8.0,NaN,6.1



▸ to_numpy(): salir de pandas hacia NumPy
[[1 'Ana' 7.5]
 [2 'Luis' 8.0]
 [3 'Eva' nan]
 [4 'Raúl' 6.1]]

  ⚠️ to_numpy() sobre columnas de tipos MEZCLADOS devuelve un array de object,
  que pierde la ventaja de velocidad de NumPy. Si vas a hacer cálculo numérico,
  selecciona primero las columnas numéricas con select_dtypes('number').


---

<a id="sec-2"></a>
## 2. Leer y escribir archivos
<sub>[↑ índice](#indice)</sub>

> **El orden correcto al abrir un archivo desconocido:** primero `nrows=1000` para
> mirar, **después** la carga completa. Cargar 89.000 registros JSON anidados para
> descubrir que la columna se llamaba distinto es tiempo perdido.

In [5]:
# Un CSV "difícil" a propósito: separador ';', coma decimal, nulos como texto
texto = """id;nombre;fecha;monto;extra
1;Ana;2026-01-05;10,5;N/A
2;Luis;2026-02-10;20,0;-
3;Eva;2026-03-15;;ok
"""

df_csv = pd.read_csv(
    StringIO(texto),             # StringIO finge ser un archivo abierto
    sep=";",                     # separador de campos (";" es común en CSV europeos)
    decimal=",",                 # la coma es el separador DECIMAL, no de campos
    na_values=["N/A", "-"],      # estos TEXTOS deben interpretarse como nulos
    parse_dates=["fecha"],       # convertir a datetime durante la lectura
    dtype={"id": "int64"},       # forzar el tipo en vez de dejar que lo infiera
    usecols=["id", "nombre", "fecha", "monto", "extra"],   # leer solo estas columnas
)
ver("CSV difícil, leído correctamente", df_csv)
ver("Los tipos quedaron bien desde la lectura", df_csv.dtypes)

# El mismo archivo SIN los parámetros, para ver qué se rompe
crudo = pd.read_csv(StringIO(texto), sep=";")
ver("Sin decimal=',' ni na_values: todo mal", crudo)
ver("'monto' quedó como TEXTO, no como número", crudo["monto"].dtype)
print("""
  Otros parámetros que vale la pena conocer:
    nrows=100          leer solo N filas (explorar rápido)
    skiprows=1         saltar filas iniciales (archivos con cabeceras de adorno)
    header=None, names=[...]   si el archivo NO trae encabezado
    encoding="latin-1" si aparecen caracteres raros (por defecto utf-8)
    on_bad_lines="skip" saltar filas rotas en vez de fallar

  Arreglar los tipos EN LA LECTURA es más barato que arreglarlos después: si
  'monto' entra como texto, cualquier operación numérica falla o, peor, hace
  concatenación de strings en silencio.""")


▸ CSV difícil, leído correctamente


,id,nombre,fecha,monto,extra
0,1,Ana,2026-01-05,10.5,NaN
1,2,Luis,2026-02-10,20.0,NaN
2,3,Eva,2026-03-15,NaN,ok



▸ Los tipos quedaron bien desde la lectura


id                 int64
nombre               str
fecha     datetime64[us]
monto            float64
extra                str
dtype: object


▸ Sin decimal=',' ni na_values: todo mal


,id,nombre,fecha,monto,extra
0,1,Ana,2026-01-05,"10,5",NaN
1,2,Luis,2026-02-10,"20,0",-
2,3,Eva,2026-03-15,NaN,ok



▸ 'monto' quedó como TEXTO, no como número
str

  Otros parámetros que vale la pena conocer:
    nrows=100          leer solo N filas (explorar rápido)
    skiprows=1         saltar filas iniciales (archivos con cabeceras de adorno)
    header=None, names=[...]   si el archivo NO trae encabezado
    encoding="latin-1" si aparecen caracteres raros (por defecto utf-8)
    on_bad_lines="skip" saltar filas rotas en vez de fallar

  Arreglar los tipos EN LA LECTURA es más barato que arreglarlos después: si
  'monto' entra como texto, cualquier operación numérica falla o, peor, hace
  concatenación de strings en silencio.


In [6]:
# --- Escribir en varios formatos y leer de vuelta ---
df.to_csv(TMP / "d.csv", index=False)       # index=False: NO escribir la columna 0,1,2...
df.to_json(TMP / "d.jsonl", orient="records", lines=True, force_ascii=False)
#   orient="records" -> una lista de objetos; lines=True -> uno por línea (JSON Lines)
#   force_ascii=False -> conserva tildes legibles en vez de á
df.to_csv(TMP / "d.csv.gz", index=False)    # la compresión se infiere de la extensión

try:
    df.to_parquet(TMP / "d.parquet")        # conserva TIPOS; requiere pyarrow
    parquet_ok = True
except ImportError:
    parquet_ok = False
    print("pyarrow no instalado: pip install pyarrow")

ver("Archivos escritos", sorted(p.name for p in TMP.iterdir()))
ver("Releído desde JSON Lines", pd.read_json(TMP / "d.jsonl", lines=True))

# La comparación que justifica usar Parquet en los intermedios
ver("Tipos originales", df.dtypes.to_dict())
ver("Tipos tras ir y volver de CSV", pd.read_csv(TMP / "d.csv").dtypes.to_dict())
if parquet_ok:
    ver("Tipos tras ir y volver de Parquet", pd.read_parquet(TMP / "d.parquet").dtypes.to_dict())
    tam = {p.suffix: p.stat().st_size for p in TMP.iterdir()}
    ver("Tamaño por formato (bytes)", tam)
print("""
  CSV pierde los tipos: todo vuelve como texto o número según lo que pandas
  adivine, y una columna de LISTAS vuelve como el string "['a', 'b']". Parquet
  los conserva y además pesa menos. Para los archivos INTERMEDIOS de tu pipeline,
  Parquet es la elección correcta; CSV solo para lo que un humano va a abrir.""")


▸ Archivos escritos
['d.csv', 'd.csv.gz', 'd.jsonl', 'd.parquet']

▸ Releído desde JSON Lines


,id,nombre,puntaje
0,1,Ana,7.5
1,2,Luis,8.0
2,3,Eva,NaN
3,4,Raúl,6.1



▸ Tipos originales
{'id': dtype('int64'), 'nombre': <StringDtype(na_value=nan)>, 'puntaje': dtype('float64')}

▸ Tipos tras ir y volver de CSV
{'id': dtype('int64'), 'nombre': <StringDtype(na_value=nan)>, 'puntaje': dtype('float64')}

▸ Tipos tras ir y volver de Parquet
{'id': dtype('int64'), 'nombre': <StringDtype(na_value=nan)>, 'puntaje': dtype('float64')}

▸ Tamaño por formato (bytes)
{'.csv': 58, '.gz': 90, '.jsonl': 156, '.parquet': 2711}

  CSV pierde los tipos: todo vuelve como texto o número según lo que pandas
  adivine, y una columna de LISTAS vuelve como el string "['a', 'b']". Parquet
  los conserva y además pesa menos. Para los archivos INTERMEDIOS de tu pipeline,
  Parquet es la elección correcta; CSV solo para lo que un humano va a abrir.


In [7]:
# --- Archivos que no caben en memoria: procesar por pedazos ---
grande = StringIO("g\n" + "\n".join(["a", "b", "a", "c"] * 1000))   # 4000 filas simuladas

conteo = Counter()                                  # acumulador PEQUEÑO (no una tabla)
for trozo in pd.read_csv(grande, chunksize=1000):   # chunksize -> iterador de DataFrames
    conteo.update(trozo["g"].value_counts().to_dict())   # agregamos el resumen del trozo
    # al cerrar la vuelta del for, `trozo` se libera: la memoria NO crece
ver("Conteo acumulado sin cargar el archivo entero", dict(conteo))

# --- Rutas portables con pathlib ---
RAIZ = Path.cwd()                      # carpeta desde la que se ejecuta el notebook
RAW = RAIZ / "datos" / "raw"           # el operador / construye rutas de forma portable
ver("Ruta construida", str(RAW))
ver("¿Existe?", RAW.exists())
print("""
  pathlib en vez de strings: "datos/raw/x.csv" falla en Windows, Path funciona
  en los tres sistemas. Y evita el bug que ya tuviste: duplicar el nombre de la
  carpeta del proyecto dentro de la ruta.

  ⚠️ La regla del chunksize: acumula SOLO el resumen. Si dentro del for haces
  lista.append(trozo) y después pd.concat(lista), volviste al problema original
  — el archivo completo en memoria, más la fragmentación.""")


▸ Conteo acumulado sin cargar el archivo entero
{'a': 2000, 'b': 1000, 'c': 1000}

▸ Ruta construida
/Users/andres/Desktop/gamecon/datos/raw

▸ ¿Existe?
True

  pathlib en vez de strings: "datos/raw/x.csv" falla en Windows, Path funciona
  en los tres sistemas. Y evita el bug que ya tuviste: duplicar el nombre de la
  carpeta del proyecto dentro de la ruta.

  ⚠️ La regla del chunksize: acumula SOLO el resumen. Si dentro del for haces
  lista.append(trozo) y después pd.concat(lista), volviste al problema original
  — el archivo completo en memoria, más la fragmentación.


### ⚠️ Troubleshooting — lectura de archivos

| Error | Causa | Arreglo |
|---|---|---|
| `UnicodeDecodeError` | el archivo no es UTF-8 | `encoding="latin-1"` o `"utf-8-sig"` |
| `ParserError: Error tokenizing data` | filas con distinto número de campos | `on_bad_lines="skip"` y revisar después |
| Una columna numérica llega como texto | separador decimal o nulos como texto | `decimal=","`, `na_values=[...]` |
| `MemoryError` | el archivo no cabe | `chunksize=`, `usecols=`, `dtype=` |
| Las fechas quedan como texto | no se pidió convertirlas | `parse_dates=["col"]` |

---

<a id="sec-3"></a>
## 3. Perfilar: conocer el dataset
<sub>[↑ índice](#indice)</sub>

**Perfilar es lo primero que se hace con un archivo que no escribiste tú**, y hay un
orden que conviene respetar porque cada pregunta condiciona la siguiente:

```
1. ¿Qué forma tiene?   shape, columns, dtypes, info
2. ¿Qué falta?         isna().sum(), isna().mean()
3. ¿Qué se repite?     duplicated(), nunique()
4. ¿Cómo se reparte?   describe(), value_counts(), quantile()
5. ¿Cuánto ocupa?      memory_usage(deep=True)
```

In [8]:
toy = pd.DataFrame({
    "nombre":  ["Alfa", "Beta", "Gamma", "Gamma", "Delta", "Epsilon", None],
    # " Acme " y "acme" son el MISMO productor escrito distinto: el problema de §9
    "empresa": [" Acme ", "acme", "Zeta Co", "Zeta Co", "Omega", None, "Omega"],
    # Tipos MEZCLADOS a propósito: int, str y NaN -> pandas deja la columna en object
    "anio":    [2001, 2005, "2010", "2010", np.nan, 1999, 2020],
    # Varios valores dentro de un solo string: la forma de vgsales y anime
    "generos": ["Action, Fantasy", "Action", "Sci-Fi, Drama, Action",
                "Sci-Fi, Drama, Action", None, "Fantasy", "Drama"],
    "ventas":  [1.5, 0.3, 9.9, 9.9, 0.1, 2.2, 0.7],
})
ver("El dataset de juguete", toy)
ver("Defectos inyectados", {"duplicadas": int(toy.duplicated().sum()),
                            "nulos": toy.isna().sum().to_dict(),
                            "dtype de 'anio'": str(toy["anio"].dtype)})


▸ El dataset de juguete


,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
1,Beta,acme,2005,Action,0.3
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
4,Delta,Omega,NaN,NaN,0.1
5,Epsilon,NaN,1999,Fantasy,2.2
6,NaN,Omega,2020,Drama,0.7



▸ Defectos inyectados
{'duplicadas': 1, 'nulos': {'nombre': 1, 'empresa': 1, 'anio': 1, 'generos': 1, 'ventas': 0}, "dtype de 'anio'": 'object'}


In [9]:
print("▸ info(): tipos, no-nulos y memoria, todo junto")
toy.info()                                     # info() IMPRIME, no devuelve: no va en ver()

ver("head(3): las primeras filas", toy.head(3))
ver("tail(2): las últimas", toy.tail(2))
ver("sample(2): al azar (random_state lo hace reproducible)", toy.sample(2, random_state=0))

ver("describe(): estadísticas de las NUMÉRICAS", toy.describe())
ver("describe(include='all'): también las de texto", toy.describe(include="all"))

ver("Nulos por columna", toy.isna().sum())
ver("Proporción de nulos", toy.isna().mean().round(2))
ver("Filas duplicadas exactas", int(toy.duplicated().sum()))
ver("Valores distintos por columna", toy.nunique())

ver("Frecuencias (incluyendo nulos)", toy["empresa"].value_counts(dropna=False))
ver("Frecuencias como proporción", toy["empresa"].value_counts(normalize=True).round(3))
ver("Varias estadísticas a la vez", toy["ventas"].agg(["min", "max", "mean", "median", "std"]))
ver("Cuartiles", toy["ventas"].quantile([.25, .5, .75]))
ver("Solo las columnas numéricas", toy.select_dtypes(include="number").columns.tolist())
print("""
  ⚠️ value_counts() DESCARTA los nulos por defecto. Si quieres saber cuántos
  registros no tienen empresa, necesitas dropna=False. Es la causa clásica del
  "me faltan filas y no sé dónde se fueron".

  ⚠️ describe() sobre 'anio' no aparece en la primera tabla porque la columna es
  object (tipos mezclados). Que una columna NO salga en describe() es, en sí,
  una señal de que hay que revisarle el tipo.""")

▸ info(): tipos, no-nulos y memoria, todo junto
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   nombre   6 non-null      str    
 1   empresa  6 non-null      str    
 2   anio     6 non-null      object 
 3   generos  6 non-null      str    
 4   ventas   7 non-null      float64
dtypes: float64(1), object(1), str(3)
memory usage: 554.0+ bytes

▸ head(3): las primeras filas


,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
1,Beta,acme,2005,Action,0.3
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9



▸ tail(2): las últimas


,nombre,empresa,anio,generos,ventas
5,Epsilon,NaN,1999,Fantasy,2.2
6,NaN,Omega,2020,Drama,0.7



▸ sample(2): al azar (random_state lo hace reproducible)


...


▸ Varias estadísticas a la vez


min       0.100000
max       9.900000
mean      3.514286
median    1.500000
std       4.420569
Name: ventas, dtype: float64


▸ Cuartiles


0.25    0.50
0.50    1.50
0.75    6.05
Name: ventas, dtype: float64


▸ Solo las columnas numéricas
['ventas']

  ⚠️ value_counts() DESCARTA los nulos por defecto. Si quieres saber cuántos
  registros no tienen empresa, necesitas dropna=False. Es la causa clásica del
  "me faltan filas y no sé dónde se fueron".

  ⚠️ describe() sobre 'anio' no aparece en la primera tabla porque la columna es
  object (tipos mezclados). Que una columna NO salga en describe() es, en sí,
  una señal de que hay que revisarle el tipo.


In [10]:
def perfil(d: pd.DataFrame, nombre: str = "dataset") -> pd.DataFrame:
    """Perfil de CUALQUIER DataFrame en una sola tabla.

    Devolver una TABLA y no prints sueltos permite ordenarla, filtrarla y
    comparar las tres fuentes del proyecto lado a lado.
    """
    print(f"== {nombre}: {d.shape[0]:,} filas × {d.shape[1]} columnas | "
          f"duplicadas: {d.duplicated().sum()} | "
          f"{d.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    return pd.DataFrame({
        "dtype":     d.dtypes.astype(str),                  # tipo como texto legible
        "nulos":     d.isna().sum(),                        # cuántos faltan
        "%nulos":    (d.isna().mean() * 100).round(1),      # y qué proporción
        "distintos": d.nunique(),                           # cardinalidad
        # cardinalidad relativa: ≈1 significa identificador; ≈0, categoría
        "card_rel":  (d.nunique() / len(d)).round(3),
        # un ejemplo real ayuda a entender el formato de un vistazo
        "ejemplo":   d.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })

display(perfil(toy, "toy"))
print("""
🎯 Para tu proyecto
   Corre perfil() sobre vgsales, anime y Goodreads, y compara:
     · candidata a PRODUCTOR -> card_rel media (cientos de valores), pocos nulos
     · candidata a GÉNERO    -> card_rel baja (decenas de valores)
     · card_rel ≈ 1.0        -> es un ID, no una categoría

   Ya mediste que 'Studios' de anime tiene 41 % de nulos. Esa cifra es una
   DECISIÓN pendiente, no un dato: ¿descartas 11.614 animes, o los agrupas en un
   stand "Estudio desconocido"? Las dos son defendibles; hay que elegir una y
   poder justificarla.""")

== toy: 7 filas × 5 columnas | duplicadas: 1 | 0.00 MB


,dtype,nulos,%nulos,distintos,card_rel,ejemplo
nombre,str,1,14.3,5,0.714,Alfa
empresa,str,1,14.3,4,0.571,Acme
anio,object,1,14.3,5,0.714,2001
generos,str,1,14.3,5,0.714,"Action, Fantasy"
ventas,float64,0,0.0,6,0.857,1.5



🎯 Para tu proyecto
   Corre perfil() sobre vgsales, anime y Goodreads, y compara:
     · candidata a PRODUCTOR -> card_rel media (cientos de valores), pocos nulos
     · candidata a GÉNERO    -> card_rel baja (decenas de valores)
     · card_rel ≈ 1.0        -> es un ID, no una categoría

   Ya mediste que 'Studios' de anime tiene 41 % de nulos. Esa cifra es una
   DECISIÓN pendiente, no un dato: ¿descartas 11.614 animes, o los agrupas en un
   stand "Estudio desconocido"? Las dos son defendibles; hay que elegir una y
   poder justificarla.


---

<a id="sec-4"></a>
## 4. Seleccionar columnas y filas
<sub>[↑ índice](#indice)</sub>

| | `.loc` | `.iloc` |
|---|---|---|
| Indexa por | **etiqueta** | **posición** |
| El final del rango | **se incluye** | **no se incluye** |

`toy.loc[0:2]` devuelve **3** filas; `toy.iloc[0:2]` devuelve **2**. No es capricho:
en un rango de etiquetas el final es parte del rango, como "de la página 10 a la 12".

In [11]:
ver("Una columna -> Series", toy["nombre"])
ver("Varias columnas -> DataFrame", toy[["nombre", "empresa"]])
ver(".loc por ETIQUETA (incluye la fila 2: son 3 filas)", toy.loc[0:2, ["nombre", "ventas"]])
ver(".iloc por POSICIÓN (excluye la 2: son 2 filas)", toy.iloc[0:2, 0:2])
ver("Filas por condición + una columna concreta", toy.loc[toy["ventas"] > 1, "nombre"])
ver(".at: acceso rápido a UNA celda (más rápido que .loc)", toy.at[0, "nombre"])
ver("iloc[-1]: la última fila", toy.iloc[-1])
ver("drop: quitar columnas (devuelve copia, no modifica)", toy.drop(columns=["ventas"]).head(3))
ver("filter(items=): columnas por nombre exacto", toy.filter(items=["nombre", "anio"]).head(3))
ver("filter(like=): el nombre contiene...", toy.filter(like="gen").head(3))
ver("filter(regex=): el nombre cumple un patrón", toy.filter(regex="^e").head(3))
ver("reindex: reordenar y/o seleccionar columnas", toy.reindex(columns=["ventas", "nombre"]).head(3))
ver("set_index + loc: usar una columna como índice", toy.set_index("nombre").loc["Beta"])
print("""
  .at vs .loc: .at solo sirve para UNA celda escalar, y por eso es mucho más
  rápido. En un bucle que lee celda a celda (que deberías evitar igual), la
  diferencia se nota.""")


▸ Una columna -> Series


0       Alfa
1       Beta
2      Gamma
3      Gamma
4      Delta
5    Epsilon
6        NaN
Name: nombre, dtype: str


▸ Varias columnas -> DataFrame


,nombre,empresa
0,Alfa,Acme
1,Beta,acme
2,Gamma,Zeta Co
3,Gamma,Zeta Co
4,Delta,Omega
5,Epsilon,NaN
6,NaN,Omega



▸ .loc por ETIQUETA (incluye la fila 2: son 3 filas)


...


▸ reindex: reordenar y/o seleccionar columnas


,ventas,nombre
0,1.5,Alfa
1,0.3,Beta
2,9.9,Gamma



▸ set_index + loc: usar una columna como índice


empresa      acme
anio         2005
generos    Action
ventas        0.3
Name: Beta, dtype: object


  .at vs .loc: .at solo sirve para UNA celda escalar, y por eso es mucho más
  rápido. En un bucle que lee celda a celda (que deberías evitar igual), la
  diferencia se nota.


---

<a id="sec-5"></a>
## 5. Filtrar con condiciones
<sub>[↑ índice](#indice)</sub>

Una condición produce una **máscara booleana** —una Series de `True`/`False`, una por
fila— y esa máscara es lo que indexa el DataFrame. Verlo así explica por qué los
paréntesis son obligatorios: `&` y `|` operan sobre las máscaras, no sobre los valores.

In [12]:
mascara = toy["ventas"] > 1
ver("La máscara booleana en sí", mascara)
ver("El DataFrame filtrado", toy[mascara])

ver("Dos condiciones con & (y)", toy[(toy["ventas"] > 1) & (toy["empresa"].notna())])
ver("Dos condiciones con | (o)", toy[(toy["ventas"] < 0.5) | (toy["ventas"] > 5)])
ver("Negación con ~", toy[~toy["empresa"].isin(["Omega"])])
ver("Pertenencia con isin", toy[toy["empresa"].isin(["Omega", "Zeta Co"])])
ver("Rango con between (extremos incluidos)", toy[toy["ventas"].between(0.5, 3)])
ver("No nulos", toy[toy["empresa"].notna()])
ver("Texto: contains (na=False es imprescindible)",
    toy[toy["generos"].str.contains("Action", na=False)])

ver("query: el filtro como texto", toy.query("ventas > 1 and empresa == 'Omega'"))
umbral = 2                                        # variable de Python
ver("query con @ para usar variables externas", toy.query("ventas > @umbral"))
ver("loc con lambda: útil dentro de cadenas de métodos", toy.loc[lambda d: d["ventas"] > 5])
ver("nlargest: top-N sin ordenar todo", toy.nlargest(2, "ventas"))
ver("where: conserva la FORMA, pone NaN donde no cumple", toy.where(toy["ventas"] > 1).head(4))
print("""
  ⚠️ El error de precedencia, que es el más frecuente del tema:
       toy[toy["ventas"] > 1 & toy["empresa"].notna()]   ← FALLA
  Python evalúa `1 & toy[...]` antes que `>`, y lanza un ValueError que no
  menciona los paréntesis por ninguna parte. Regla: cada condición entre
  paréntesis, sin excepciones.

  ⚠️ Comparar con NaN: x == np.nan es SIEMPRE False, incluso para NaN == NaN.
  Para nulos se usa isna() / notna(), nunca ==.

  filtrado vs where: el filtrado DEVUELVE menos filas; where devuelve TODAS las
  filas con NaN donde no se cumple. Si necesitas conservar la forma para una
  operación posterior, where es el correcto.""")


▸ La máscara booleana en sí


0     True
1    False
2     True
3     True
4    False
5     True
6    False
Name: ventas, dtype: bool


▸ El DataFrame filtrado


,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
5,Epsilon,NaN,1999,Fantasy,2.2



▸ Dos condiciones con & (y)


...


▸ nlargest: top-N sin ordenar todo


,nombre,empresa,anio,generos,ventas
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9



▸ where: conserva la FORMA, pone NaN donde no cumple


,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
1,NaN,NaN,NaN,NaN,NaN
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9



  ⚠️ El error de precedencia, que es el más frecuente del tema:
       toy[toy["ventas"] > 1 & toy["empresa"].notna()]   ← FALLA
  Python evalúa `1 & toy[...]` antes que `>`, y lanza un ValueError que no
  menciona los paréntesis por ninguna parte. Regla: cada condición entre
  paréntesis, sin excepciones.

  ⚠️ Comparar con NaN: x == np.nan es SIEMPRE False, incluso para NaN == NaN.
  Para nulos se usa isna() / notna(), nunca ==.

  filtrado vs where: el filtrado DEVUELVE menos filas; where devuelve TODAS las
  filas con NaN donde no se cumple. Si necesitas conservar la forma para una
  operación posterior, where es el correcto.


---

<a id="parte-b"></a>
# Parte B · Limpieza

<a id="sec-6"></a>
## 6. Valores nulos
<sub>[↑ índice](#indice)</sub>

> ⚠️ **Antes de limpiar, decide qué significa "limpio".** Las reglas de limpieza salen
> del **modelo destino**, no al revés. Si todavía no sabes si un stand es la editorial
> o la revista, no sabes cuál es tu `stand_id`, y por lo tanto no puedes decidir si una
> fila sin editorial se descarta o va a "Independiente". Esa decisión es del grupo y va
> **antes** del código.

In [13]:
n = pd.DataFrame({"a": [1, np.nan, 3, None],       # None y np.nan son ambos nulos aquí
                  "b": ["x", None, "z", "w"],
                  "c": [np.nan] * 4})              # columna completamente vacía
ver("Datos con nulos", n)
ver("isna(): máscara de nulos", n.isna())
ver("notna(): la negación", n.notna())

ver("dropna(): fuera TODA fila con algún nulo", n.dropna())
ver("dropna(how='all'): solo filas TOTALMENTE vacías", n.dropna(how="all"))
ver("dropna(subset=['b']): mirando una sola columna", n.dropna(subset=["b"]))
ver("dropna(axis=1, how='all'): quitar COLUMNAS vacías", n.dropna(axis=1, how="all"))

ver("fillna con dict: relleno distinto por columna", n.fillna({"a": 0, "b": "desconocido"}))
ver("fillna con un estadístico", n["a"].fillna(n["a"].mean()))
ver("ffill: propaga el valor ANTERIOR", n["a"].ffill())
ver("bfill: propaga el SIGUIENTE", n["a"].bfill())
ver("interpolate: interpola numéricos", n["a"].interpolate())
ver("replace: convertir un valor EN nulo", n.replace({"x": np.nan}))
print("""
  Cuál usar no es cuestión de gusto:
    dropna      -> cuando la fila sin ese dato no sirve para nada
    fillna(0)   -> solo si 0 es un valor con SENTIDO (no para un precio)
    fillna(media)-> imputación; SIEMPRE deja una columna marca "fue_imputado"
    ffill/bfill -> series temporales, donde "lo último conocido" tiene sentido
    interpolate -> magnitudes continuas medidas a intervalos

  ⚠️ Un dato imputado que no se declara es un dato falso. En tres semanas nadie
  va a recordar qué precios eran reales.""")


▸ Datos con nulos


,a,b,c
0,1.0,x,NaN
1,NaN,NaN,NaN
2,3.0,z,NaN
3,NaN,w,NaN



▸ isna(): máscara de nulos


,a,b,c
0,False,False,True
1,True,True,True
2,False,False,True
3,True,False,True



▸ notna(): la negación


...


▸ interpolate: interpola numéricos


0    1.0
1    2.0
2    3.0
3    3.0
Name: a, dtype: float64


▸ replace: convertir un valor EN nulo


,a,b,c
0,1.0,NaN,NaN
1,NaN,NaN,NaN
2,3.0,z,NaN
3,NaN,w,NaN



  Cuál usar no es cuestión de gusto:
    dropna      -> cuando la fila sin ese dato no sirve para nada
    fillna(0)   -> solo si 0 es un valor con SENTIDO (no para un precio)
    fillna(media)-> imputación; SIEMPRE deja una columna marca "fue_imputado"
    ffill/bfill -> series temporales, donde "lo último conocido" tiene sentido
    interpolate -> magnitudes continuas medidas a intervalos

  ⚠️ Un dato imputado que no se declara es un dato falso. En tres semanas nadie
  va a recordar qué precios eran reales.


---

<a id="sec-7"></a>
## 7. Tipos de datos
<sub>[↑ índice](#indice)</sub>

`errors="coerce"` es el parámetro que convierte un script frágil en uno auditable: lo
que no se puede convertir pasa a `NaN` en vez de lanzar una excepción, y así **puedes
contarlo** en vez de que te detenga.

In [14]:
t = pd.DataFrame({"n": ["1", "2", "x", None],                         # números como texto
                  "f": ["2026-01-01", "2026-02-01", "mal", None],     # fechas como texto
                  "c": ["a", "b", "a", "a"]})                         # categoría

t["n_num"] = pd.to_numeric(t["n"], errors="coerce")      # lo inválido -> NaN (no revienta)
t["f_dt"]  = pd.to_datetime(t["f"], errors="coerce")     # lo inválido -> NaT
t["c_cat"] = t["c"].astype("category")                   # pocos valores distintos -> ahorra memoria
t["n_int"] = t["n_num"].astype("Int64")                  # Int64 con MAYÚSCULA: entero CON nulos
t["s"]     = t["c"].astype("string")                     # tipo texto explícito de pandas
t["bool"]  = t["c"].eq("a")                              # comparación -> columna booleana

ver("Resultado", t)
ver("Tipos finales", t.dtypes)
ver("¿Cuántos valores no se pudieron convertir?",
    {"n -> número": int(t["n_num"].isna().sum()), "f -> fecha": int(t["f_dt"].isna().sum())})
ver("convert_dtypes(): pandas elige los mejores tipos solo", t.convert_dtypes().dtypes)

# La diferencia que importa entre int64 e Int64
ver("Sin Int64, el entero se vuelve float y el año se ve feo",
    pd.Series([2001, None]).tolist())
ver("Con Int64 se mantiene entero", pd.Series([2001, None], dtype="Int64").tolist())
print("""
  ⚠️ errors="coerce" vs el comportamiento por defecto:
     sin él     -> lanza ValueError y el script muere en la fila 8.432
     con él     -> deja NaN, y tú decides: ¿cuántos son?, ¿los descarto o los reparo?
  Lo segundo es auditable; lo primero, no.""")


▸ Resultado


,n,f,c,n_num,f_dt,c_cat,n_int,s,bool
0,1,2026-01-01,a,1.0,2026-01-01,a,1,a,True
1,2,2026-02-01,b,2.0,2026-02-01,b,2,b,False
2,x,mal,a,NaN,NaT,a,<NA>,a,True
3,NaN,NaN,a,NaN,NaT,a,<NA>,a,True



▸ Tipos finales


n                   str
f                   str
c                   str
n_num           float64
f_dt     datetime64[us]
c_cat          category
n_int             Int64
s                string
dtype: object


▸ ¿Cuántos valores no se pudieron convertir?
{'n -> número': 2, 'f -> fecha': 2}

▸ convert_dtypes(): pandas elige los mejores tipos solo


n                string
f                string
c                string
n_num             Int64
f_dt     datetime64[us]
c_cat          category
n_int             Int64
s                string
dtype: object


▸ Sin Int64, el entero se vuelve float y el año se ve feo
[2001.0, nan]

▸ Con Int64 se mantiene entero
[2001, <NA>]

  ⚠️ errors="coerce" vs el comportamiento por defecto:
     sin él     -> lanza ValueError y el script muere en la fila 8.432
     con él     -> deja NaN, y tú decides: ¿cuántos son?, ¿los descarto o los reparo?
  Lo segundo es auditable; lo primero, no.


---

<a id="sec-8"></a>
## 8. Duplicados
<sub>[↑ índice](#indice)</sub>

`keep` decide **cuál** se conserva, y la elección importa: si tu CSV viene ordenado por
fecha, `keep="last"` conserva la versión más reciente y `keep="first"` la más antigua.

In [15]:
ver("duplicated(): True en la SEGUNDA aparición", toy.duplicated())
ver("duplicated(keep=False): marca TODAS las copias", toy.duplicated(keep=False))
ver("duplicated(subset=): según columnas concretas", toy.duplicated(subset=["nombre"]))

ver("drop_duplicates(): quita filas idénticas", toy.drop_duplicates())
ver("drop_duplicates(subset, keep='last'): conserva la última",
    toy.drop_duplicates(subset=["nombre"], keep="last"))

# El paso que casi nadie hace y debería: MIRAR qué se repite antes de borrar
ver("Inspeccionar los duplicados antes de eliminarlos",
    toy[toy.duplicated(subset=["nombre"], keep=False)].sort_values("nombre"))
ver("Cuántas filas se perderían", {"antes": len(toy), "después": len(toy.drop_duplicates())})
print("""
  ⚠️ Un duplicado EXACTO (todas las columnas iguales) casi siempre es un error de
  carga y se puede borrar. Un duplicado PARCIAL (mismo nombre, distintas ventas)
  puede ser información legítima: el mismo juego en dos plataformas. Antes de
  usar subset= hay que saber qué significa una fila en ese dataset.""")


▸ duplicated(): True en la SEGUNDA aparición


0    False
1    False
2    False
3     True
4    False
5    False
6    False
dtype: bool


▸ duplicated(keep=False): marca TODAS las copias


0    False
1    False
2     True
3     True
4    False
5    False
6    False
dtype: bool


▸ duplicated(subset=): según columnas concretas


...


▸ drop_duplicates(subset, keep='last'): conserva la última


,nombre,empresa,anio,generos,ventas
0,Alfa,Acme,2001,"Action, Fantasy",1.5
1,Beta,acme,2005,Action,0.3
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
4,Delta,Omega,NaN,NaN,0.1
5,Epsilon,NaN,1999,Fantasy,2.2
6,NaN,Omega,2020,Drama,0.7



▸ Inspeccionar los duplicados antes de eliminarlos


,nombre,empresa,anio,generos,ventas
2,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9
3,Gamma,Zeta Co,2010,"Sci-Fi, Drama, Action",9.9



▸ Cuántas filas se perderían
{'antes': 7, 'después': 6}

  ⚠️ Un duplicado EXACTO (todas las columnas iguales) casi siempre es un error de
  carga y se puede borrar. Un duplicado PARCIAL (mismo nombre, distintas ventas)
  puede ser información legítima: el mismo juego en dos plataformas. Antes de
  usar subset= hay que saber qué significa una fila en ese dataset.


---

<a id="sec-9"></a>
## 9. Texto con `.str`
<sub>[↑ índice](#indice)</sub>

### Por qué esta sección decide si el recomendador sirve

`"Acme"`, `" Acme "` y `"acme"` son **tres productores distintos** para pandas. Sin
unificarlos antes de agrupar, tu feria tendrá tres stands donde debe haber uno, con las
etiquetas repartidas entre los tres.

In [16]:
tx = pd.Series(["  Role-Playing ", "SCI-FI", "Slice of Life", None,
                "Sports/Racing", "Acción y Aventura"])
ver("Serie original (ojo al None)", tx)

ver("strip: quita espacios de los bordes", tx.str.strip().tolist())
ver("lower: minúsculas", tx.str.lower().tolist())
ver("len: largo (NaN se propaga)", tx.str.len().tolist())
ver("replace literal (regex=False)", tx.str.replace("-", " ", regex=False).tolist())
ver("replace con regex: colapsar espacios", tx.str.replace(r"\s+", " ", regex=True).tolist())
ver("contains (na=False evita el error al filtrar)",
    tx.str.contains("sci", case=False, na=False).tolist())
ver("startswith / endswith",
    (tx.str.startswith("Sl", na=False).tolist(), tx.str.endswith("ng", na=False).tolist()))
ver("split: texto -> lista", tx.str.split("/").tolist())
ver("split(expand=True): texto -> COLUMNAS", tx.str.split(" ", expand=True))
ver("extract: grupos de regex -> columnas", tx.str.extract(r"(\w+)-(\w+)"))
ver("findall: todas las coincidencias", tx.str.findall(r"[A-Z]").tolist())
ver("slicing con .str[:4]", tx.str[:4].tolist())
ver("cat: unir toda la columna en un texto", tx.str.cat(sep=" | ", na_rep="∅"))
ver("zfill / pad: rellenar a un ancho fijo",
    (tx.str.zfill(18).head(2).tolist(), tx.str.pad(20, fillchar="·").head(2).tolist()))
print("""
  ⚠️ na=False en contains/startswith NO es opcional cuando filtras. Sin él los
  nulos producen NaN en lugar de False, y pandas no puede indexar con NaN:
  "Cannot mask with non-boolean array containing NA".

  ⚠️ regex=True vs False: replace("-", " ") con regex=True trata el guion como
  parte de una expresión regular. Con caracteres como . * + ? [ ] ( ) la
  diferencia deja de ser cosmética y cambia el resultado.""")


▸ Serie original (ojo al None)


0        Role-Playing 
1               SCI-FI
2        Slice of Life
3                  NaN
4        Sports/Racing
5    Acción y Aventura
dtype: str


▸ strip: quita espacios de los bordes
['Role-Playing', 'SCI-FI', 'Slice of Life', nan, 'Sports/Racing', 'Acción y Aventura']

▸ lower: minúsculas
['  role-playing ', 'sci-fi', 'slice of life', nan, 'sports/racing', 'acción y aventura']

▸ len: largo (NaN se propaga)
[15.0, 6.0, 13.0, nan, 13.0, 17.0]

▸ replace literal (regex=False)
['  Role Playing ', 'SCI FI', 'Slice of Life', nan, 'Sports/Racing', 'Acción y Aventura']

▸ replace con regex: colapsar espacios
[' Role-Playing ', 'SCI-FI', 'Slice of Life', nan, 'Sports/Racing', 'Acción y Aventura']

▸ contains (na=False evita el error al filtrar)
[False, True, False, False, False, False]

▸ startswith / endswith
([False, False, True, False, False, False], [False, False, False, False, True, False])

▸ split: texto -> lista
[['  Role-Playing '], ['SCI-FI'], ['Slice of Life'], nan, ['Sports', 'Racing'], ['Acción y Aventura']]

▸ split(expand=True): texto -> COLUMNAS


,0,1,2,3
0,,,Role-Playing,
1,SCI-FI,NaN,NaN,NaN
2,Slice,of,Life,NaN
3,NaN,NaN,NaN,NaN
4,Sports/Racing,NaN,NaN,NaN
5,Acción,y,Aventura,NaN



▸ extract: grupos de regex -> columnas


,0,1
0,Role,Playing
1,SCI,FI
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN



▸ findall: todas las coincidencias
[['R', 'P'], ['S', 'C', 'I', 'F', 'I'], ['S', 'L'], nan, ['S', 'R'], ['A', 'A']]

▸ slicing con .str[:4]
['  Ro', 'SCI-', 'Slic', nan, 'Spor', 'Acci']

▸ cat: unir toda la columna en un texto
  Role-Playing  | SCI-FI | Slice of Life | ∅ | Sports/Racing | Acción y Aventura

▸ zfill / pad: rellenar a un ancho fijo
(['000  Role-Playing ', '000000000000SCI-FI'], ['·····  Role-Playing ', '··············SCI-FI'])

  ⚠️ na=False en contains/startswith NO es opcional cuando filtras. Sin él los
  nulos producen NaN en lugar de False, y pandas no puede indexar con NaN:
  "Cannot mask with non-boolean array containing NA".

  ⚠️ regex=True vs False: replace("-", " ") con regex=True trata el guion como
  parte de una expresión regular. Con caracteres como . * + ? [ ] ( ) la
  diferencia deja de ser cosmética y cambia el resultado.


In [17]:
def quitar_tildes(x):
    """'Acción' -> 'Accion', sin lista de reemplazos a mano.

    NFKD descompone cada letra acentuada en (letra base + marca de acento) y
    unicodedata.combining() identifica esas marcas para descartarlas. Hacerlo
    con .replace("á","a") obliga a enumerar todos los casos y siempre falta uno.
    """
    if not isinstance(x, str):          # nulos y números pasan de largo sin romper
        return x
    descompuesto = unicodedata.normalize("NFKD", x)       # 'ó' -> 'o' + marca
    return "".join(c for c in descompuesto if not unicodedata.combining(c))


def normalizar(serie: pd.Series) -> pd.Series:
    """La función que debe aplicarse a TODA etiqueta antes de compararla.

    El orden no es arbitrario: si reemplazaras separadores antes de quitar
    tildes, 'Ciencia-Ficción' y 'ciencia ficcion' no colapsarían en lo mismo.
    """
    return (serie.map(quitar_tildes)                           # 1. fuera tildes
                 .str.strip()                                   # 2. fuera espacios de borde
                 .str.lower()                                   # 3. minúsculas
                 # 4. TODO separador (espacio, _, /, .) pasa a ser UN guion.
                 #    Incluir el punto es lo que hace que 'ciencia.ficcion'
                 #    colapse con 'ciencia-ficcion' en vez de quedar aparte.
                 .str.replace(r"[\s_/.]+", "-", regex=True)
                 .str.replace(r"[^a-z0-9\-]", "", regex=True)   # 5. fuera el resto del ruido
                 .str.replace(r"-+", "-", regex=True)           # 6. colapsar guiones repetidos
                 .str.strip("-"))                               # 7. sin guiones en los bordes


ver("normalizar() sobre la serie de ejemplo", normalizar(tx).tolist())

variantes = pd.Series(["Ciencia-Ficción", "ciencia ficcion", "  CIENCIA   FICCION  ",
                       "Ciencia_Ficción", "ciencia.ficción"])
norm = normalizar(variantes)
ver("5 formas de escribir lo mismo", norm.tolist())
ver("¿colapsan todas en UNA?", norm.nunique() == 1)

# La COMPARACIÓN que justifica el paso 4. Versión sin el punto en los separadores:
sin_punto = (variantes.map(quitar_tildes).str.strip().str.lower()
             .str.replace(r"[\s_/]+", "-", regex=True)        # el punto NO es separador
             .str.replace(r"[^a-z0-9\-]", "", regex=True)
             .str.strip("-"))
ver("Sin el punto entre los separadores", {"resultados": sorted(set(sin_punto)),
                                           "distintos": sin_punto.nunique()})
print("""
  Ahí está la lección, y es sutil: en la versión de abajo el punto NO cae en el
  paso 4 (separadores -> guion), así que llega al paso 5 (borrar ruido) y se
  BORRA. 'ciencia.ficcion' se vuelve 'cienciaficcion', que no es igual a
  'ciencia-ficcion'. Resultado: 2 etiquetas donde debía haber 1.

  La diferencia entre "normalizar un separador" y "borrar un carácter" decide si
  dos etiquetas colapsan o no. Qué caracteres cuentan como separador en TUS
  datos es una decisión que hay que tomar mirándolos, no copiando una regex.

🎯 Para tu proyecto
   Escribe TU normalizar() y pásala por todas las etiquetas de las tres fuentes.
   Después cuenta: ¿cuántas etiquetas distintas había antes y cuántas quedan?
   Si el número no baja, la normalización no está haciendo nada.""")


▸ normalizar() sobre la serie de ejemplo
['role-playing', 'sci-fi', 'slice-of-life', nan, 'sports-racing', 'accion-y-aventura']

▸ 5 formas de escribir lo mismo
['ciencia-ficcion', 'ciencia-ficcion', 'ciencia-ficcion', 'ciencia-ficcion', 'ciencia-ficcion']

▸ ¿colapsan todas en UNA?
True

▸ Sin el punto entre los separadores
{'resultados': ['ciencia-ficcion', 'cienciaficcion'], 'distintos': 2}

  Ahí está la lección, y es sutil: en la versión de abajo el punto NO cae en el
  paso 4 (separadores -> guion), así que llega al paso 5 (borrar ruido) y se
  BORRA. 'ciencia.ficcion' se vuelve 'cienciaficcion', que no es igual a
  'ciencia-ficcion'. Resultado: 2 etiquetas donde debía haber 1.

  La diferencia entre "normalizar un separador" y "borrar un carácter" decide si
  dos etiquetas colapsan o no. Qué caracteres cuentan como separador en TUS
  datos es una decisión que hay que tomar mirándolos, no copiando una regex.

🎯 Para tu proyecto
   Escribe TU normalizar() y pásala por todas las e

---

<a id="sec-10"></a>
## 10. Fechas y horas
<sub>[↑ índice](#indice)</sub>

El accesor `.dt` es a las fechas lo que `.str` es al texto: aplica la operación a toda
la columna. Lo necesitas para las **presentaciones del escenario**, que tienen horario.

In [18]:
f = pd.to_datetime(pd.Series(["2026-11-01 10:00", "2026-11-01 12:30", "2026-11-02 09:15"]))
ver("Serie de fechas", f)

ver(".dt.date: solo la fecha", f.dt.date.tolist())
ver(".dt.hour: la hora", f.dt.hour.tolist())
ver(".dt.day_name(): el día de la semana", f.dt.day_name().tolist())
ver("Sumar una duración con Timedelta", (f + pd.Timedelta(hours=1, minutes=30)).tolist())
ver("Restar dos instantes -> Timedelta", f.iloc[1] - f.iloc[0])
ver("floor('h'): redondear hacia abajo a la hora", f.dt.floor("h").tolist())
ver("date_range: una secuencia regular", pd.date_range("2026-11-01 10:00", periods=4, freq="2h").tolist())
ver("Formato explícito cuando es ambiguo", pd.to_datetime("01/11/2026", format="%d/%m/%Y"))
ver("Sumar varios desfases a la vez",
    pd.Timestamp("2026-11-01 10:00") + pd.to_timedelta(np.array([30, 90]), unit="m"))

# El filtro que necesitas para las presentaciones: ¿cuáles caen en una ventana?
ini, fin = pd.Timestamp("2026-11-01 11:00"), pd.Timestamp("2026-11-01 13:00")
ver("Eventos dentro de una ventana horaria", f[(f >= ini) & (f <= fin)])
print("""
  ⚠️ format="%d/%m/%Y" no es opcional con fechas ambiguas: "01/11/2026" es el 1
  de noviembre en Chile y el 11 de enero en EE.UU. Sin format, pandas adivina —
  y a veces adivina distinto para distintas filas del mismo archivo.""")


▸ Serie de fechas


0   2026-11-01 10:00:00
1   2026-11-01 12:30:00
2   2026-11-02 09:15:00
dtype: datetime64[us]


▸ .dt.date: solo la fecha
[datetime.date(2026, 11, 1), datetime.date(2026, 11, 1), datetime.date(2026, 11, 2)]

▸ .dt.hour: la hora
[10, 12, 9]

▸ .dt.day_name(): el día de la semana
['Sunday', 'Sunday', 'Monday']

▸ Sumar una duración con Timedelta
[Timestamp('2026-11-01 11:30:00'), Timestamp('2026-11-01 14:00:00'), Timestamp('2026-11-02 10:45:00')]

▸ Restar dos instantes -> Timedelta
0 days 02:30:00

▸ floor('h'): redondear hacia abajo a la hora
[Timestamp('2026-11-01 10:00:00'), Timestamp('2026-11-01 12:00:00'), Timestamp('2026-11-02 09:00:00')]

▸ date_range: una secuencia regular
[Timestamp('2026-11-01 10:00:00'), Timestamp('2026-11-01 12:00:00'), Timestamp('2026-11-01 14:00:00'), Timestamp('2026-11-01 16:00:00')]

▸ Formato explícito cuando es ambiguo
2026-11-01 00:00:00

▸ Sumar varios desfases a la vez
DatetimeIndex(['2026-11-01 10:30:00', '2026-11-01 11:30:00'], dtype='datetime64[us]', freq=None)

▸ Eventos dentro de una ventana horaria


1   2026-11-01 12:30:00
dtype: datetime64[us]


  ⚠️ format="%d/%m/%Y" no es opcional con fechas ambiguas: "01/11/2026" es el 1
  de noviembre en Chile y el 11 de enero en EE.UU. Sin format, pandas adivina —
  y a veces adivina distinto para distintas filas del mismo archivo.


---

<a id="sec-11"></a>
## 11. Aplicar funciones: `map`, `apply`, `assign`, `pipe`
<sub>[↑ índice](#indice)</sub>

| Herramienta | Qué recorre | Velocidad |
|---|---|---|
| `np.where` / `np.select` | vectorizado | **la más rápida** |
| `.map` | elemento a elemento de una Series | media |
| `.apply(axis=0)` | columna a columna | media |
| `.apply(axis=1)` | **fila a fila** | **la más lenta** — evítala |

> **Regla.** Si existe una versión vectorizada, úsala. `apply(axis=1)` sobre 200.000
> filas puede tardar **mil veces más** que la operación equivalente con columnas, y en
> §23 lo medimos.

In [19]:
d = pd.DataFrame({"a": [1, 2, 3, 4], "b": [10, 20, 30, 40], "g": ["x", "y", "x", "z"]})

ver("map con dict: lo NO encontrado -> NaN", d["g"].map({"x": "X", "y": "Y"}).tolist())
ver("map + fillna: dejar el original si no está en el dict",
    d["g"].map({"x": "X"}).fillna(d["g"]).tolist())
ver("apply sobre una Series: elemento a elemento", d["a"].apply(lambda v: v ** 2).tolist())
ver("apply(axis=1): fila a fila (LENTO)", d.apply(lambda fila: fila["a"] + fila["b"], axis=1).tolist())
ver("apply sobre un DataFrame: columna a columna", d[["a", "b"]].apply(np.sum))

# assign + pipe: encadenar sin crear variables intermedias ni mutar el original
d2 = (d.assign(suma=lambda x: x["a"] + x["b"])        # assign AÑADE columnas y devuelve copia
        .pipe(lambda x: x[x["suma"] > 15])            # pipe mete una función propia en la cadena
        .rename(columns={"g": "grupo"}))
ver("Cadena assign -> pipe -> rename", d2)
ver("El original NO se modificó", d.columns.tolist())

# Condiciones vectorizadas: preferirlas SIEMPRE sobre apply
d["tam"] = np.where(d["a"] > 2, "grande", "chico")                 # if/else vectorizado
d["nivel"] = np.select([d["a"] < 2, d["a"] < 4],                   # lista de condiciones
                       ["bajo", "medio"], default="alto")          # valor para cada una
ver("np.where y np.select", d)
ver("clip: recortar a un rango", d["a"].clip(lower=2, upper=3).tolist())
ver("cut: tramos de ANCHO fijo que tú defines",
    pd.cut(d["b"], bins=[0, 15, 35, 100], labels=["bajo", "medio", "alto"]).tolist())
ver("qcut: tramos de igual CANTIDAD de elementos (cuantiles)",
    pd.qcut(d["b"], q=2, labels=["mitad baja", "mitad alta"]).tolist())
print("""
  cut vs qcut, que se confunden:
    cut  -> tú defines los CORTES. Tramos de ancho fijo, cantidades desiguales.
    qcut -> pandas define los cortes para que cada tramo tenga el MISMO número
            de elementos. Útil para percentiles; inútil si los cortes tienen
            que significar algo concreto (precio barato/caro).""")


▸ map con dict: lo NO encontrado -> NaN
['X', 'Y', 'X', nan]

▸ map + fillna: dejar el original si no está en el dict
['X', 'y', 'X', 'z']

▸ apply sobre una Series: elemento a elemento
[1, 4, 9, 16]

▸ apply(axis=1): fila a fila (LENTO)
[11, 22, 33, 44]

▸ apply sobre un DataFrame: columna a columna


a     10
b    100
dtype: int64


▸ Cadena assign -> pipe -> rename


,a,b,grupo,suma
1,2,20,y,22
2,3,30,x,33
3,4,40,z,44



▸ El original NO se modificó
['a', 'b', 'g']

▸ np.where y np.select


,a,b,g,tam,nivel
0,1,10,x,chico,bajo
1,2,20,y,chico,medio
2,3,30,x,grande,medio
3,4,40,z,grande,alto



▸ clip: recortar a un rango
[2, 2, 3, 3]

▸ cut: tramos de ANCHO fijo que tú defines
['bajo', 'medio', 'medio', 'alto']

▸ qcut: tramos de igual CANTIDAD de elementos (cuantiles)
['mitad baja', 'mitad baja', 'mitad alta', 'mitad alta']

  cut vs qcut, que se confunden:
    cut  -> tú defines los CORTES. Tramos de ancho fijo, cantidades desiguales.
    qcut -> pandas define los cortes para que cada tramo tenga el MISMO número
            de elementos. Útil para percentiles; inútil si los cortes tienen
            que significar algo concreto (precio barato/caro).


---

<a id="parte-c"></a>
# Parte C · Transformar

<a id="sec-12"></a>
## 12. Columnas con varios valores
<sub>[↑ índice](#indice)</sub>

```
FORMA ANCHA (como viene)            FORMA LARGA (para contar y cruzar)
┌───────┬─────────────────────┐     ┌───────┬──────────┐
│ Gamma │ Sci-Fi, Drama,      │     │ Gamma │ sci-fi   │
│       │ Action              │  →  │ Gamma │ drama    │
└───────┴─────────────────────┘     │ Gamma │ action   │
  1 fila, 3 valores en un string    └───────┴──────────┘
```

`explode` es el puente de ida; `groupby(...).agg(sorted(set(...)))` el de vuelta.

In [20]:
g = toy[["nombre", "generos"]].dropna().drop_duplicates().copy()
g["lista"] = g["generos"].str.split(",")                               # texto -> lista
g["lista"] = g["lista"].map(lambda xs: [x.strip().lower() for x in xs])  # limpiar cada elemento
ver("Columna de listas", g[["nombre", "lista"]])

# LA MEDICIÓN QUE DECIDE SI JACCARD SIRVE
n_tags = g["lista"].map(len)
ver("Etiquetas por ítem", {"mediana": float(n_tags.median()),
                           "máximo": int(n_tags.max()), "mínimo": int(n_tags.min())})

largo = g.explode("lista").rename(columns={"lista": "genero"})   # una fila por elemento
ver("Forma larga (índice REPETIDO: es normal)", largo[["nombre", "genero"]])

# El camino de vuelta: de larga a listas sin repetidos y ordenadas
ver("Reagrupar a listas",
    largo.groupby("nombre")["genero"].agg(lambda s: sorted(set(s))).reset_index())
print("""
  Si la MEDIANA de etiquetas por ítem es 1, Jaccard entre dos ítems solo puede
  valer 0 o 1: deja de medir parecido y pasa a medir igualdad. Ya lo mediste en
  vgsales (Genre tiene UN valor por fila) y en anime (Genres+Themes dan mediana
  2). Esa diferencia entre fuentes es un problema de diseño del proyecto, no un
  detalle técnico.""")


▸ Columna de listas


,nombre,lista
0,Alfa,"[action, fantasy]"
1,Beta,[action]
2,Gamma,"[sci-fi, drama, action]"
5,Epsilon,[fantasy]



▸ Etiquetas por ítem
{'mediana': 1.5, 'máximo': 3, 'mínimo': 1}

▸ Forma larga (índice REPETIDO: es normal)


,nombre,genero
0,Alfa,action
0,Alfa,fantasy
1,Beta,action
2,Gamma,sci-fi
2,Gamma,drama
2,Gamma,action
5,Epsilon,fantasy



▸ Reagrupar a listas


,nombre,genero
0,Alfa,"[action, fantasy]"
1,Beta,[action]
2,Epsilon,[fantasy]
3,Gamma,"[action, drama, sci-fi]"



  Si la MEDIANA de etiquetas por ítem es 1, Jaccard entre dos ítems solo puede
  valer 0 o 1: deja de medir parecido y pasa a medir igualdad. Ya lo mediste en
  vgsales (Genre tiene UN valor por fila) y en anime (Genres+Themes dan mediana
  2). Esa diferencia entre fuentes es un problema de diseño del proyecto, no un
  detalle técnico.


In [21]:
ver("get_dummies sobre el string original", g["generos"].str.get_dummies(sep=", "))
ver("get_dummies sobre la forma larga", pd.get_dummies(largo["genero"], dtype=int))

# Listas guardadas como TEXTO (típico al releer un CSV)
ver("ast.literal_eval convierte el texto en lista real",
    pd.Series(["['a', 'b']", "['c']"]).map(ast.literal_eval).tolist())

# Unir varias columnas de listas en una sola: el caso Genres + Themes + Demographics
x = pd.DataFrame({"gen": [["a", "b"], ["c"]], "tem": [["d"], []], "dem": [["e"], ["f"]]})
x["todo"] = x["gen"] + x["tem"] + x["dem"]                  # + concatena listas fila a fila
x["todo"] = x["todo"].map(lambda xs: sorted(set(xs)))       # sin repetidos y determinista
ver("Tres columnas de listas unidas en una", x)
ver("Número de elementos por fila", x["todo"].map(len).tolist())
ver("Aplanar y contar TODOS los elementos",
    pd.Series([e for lst in x["todo"] for e in lst]).value_counts())

# JSON anidado: dicts dentro de una columna
anid = pd.DataFrame({"id": [1, 2], "info": [{"x": 1, "y": {"z": 9}}, {"x": 3, "y": {"z": 7}}]})
ver("json_normalize: el anidado se aplana con punto", pd.json_normalize(anid["info"]))
ver("record_path: una fila por elemento de la lista interna",
    pd.json_normalize([{"id": 1, "libros": [{"t": "A"}, {"t": "B"}]}],
                      record_path="libros", meta=["id"]))
print("""
  ⚠️ NUNCA uses eval() para convertir texto en objeto. ast.literal_eval solo
  acepta literales de Python y por eso es seguro; eval ejecuta CUALQUIER código
  que venga en el archivo.

🎯 Para tu proyecto
   popular_shelves de Goodreads es una lista de dicts que MEZCLA géneros con
   ruido de usuarios ('to-read', 'currently-reading', 'favorites'). Antes de
   explotarla hay que decidir el criterio: ¿conteo mínimo?, ¿lista blanca? Es
   una decisión de diseño, y hay que poder defenderla.""")


▸ get_dummies sobre el string original


,Action,Drama,Fantasy,Sci-Fi
0,1,0,1,0
1,1,0,0,0
2,1,1,0,1
5,0,0,1,0



▸ get_dummies sobre la forma larga


,action,drama,fantasy,sci-fi
0,1,0,0,0
0,0,0,1,0
1,1,0,0,0
2,0,0,0,1
2,0,1,0,0
2,1,0,0,0
5,0,0,1,0



▸ ast.literal_eval convierte el texto en lista real
[['a', 'b'], ['c']]

▸ Tres columnas de listas unidas en una


...


▸ json_normalize: el anidado se aplana con punto


,x,y.z
0,1,9
1,3,7



▸ record_path: una fila por elemento de la lista interna


,t,id
0,A,1
1,B,1



  ⚠️ NUNCA uses eval() para convertir texto en objeto. ast.literal_eval solo
  acepta literales de Python y por eso es seguro; eval ejecuta CUALQUIER código
  que venga en el archivo.

🎯 Para tu proyecto
   popular_shelves de Goodreads es una lista de dicts que MEZCLA géneros con
   ruido de usuarios ('to-read', 'currently-reading', 'favorites'). Antes de
   explotarla hay que decidir el criterio: ¿conteo mínimo?, ¿lista blanca? Es
   una decisión de diseño, y hay que poder defenderla.


---

<a id="sec-13"></a>
## 13. Mapeo con diccionarios y tablas de equivalencia
<sub>[↑ índice](#indice)</sub>

Normalizar (§9) resuelve la **ortografía**; no resuelve el **vocabulario**. `"sci-fi"`,
`"science fiction"` y `"cf"` pueden estar los tres bien normalizados y seguir siendo
tres etiquetas distintas.

> **Idea clave, y es de diseño.** La tabla de equivalencias es un **artefacto humano**:
> un CSV que una persona del grupo edita y que el código solo lee. Así Cris y Sofi
> pueden discutirla sin tocar Python, y en la defensa puedes **mostrarla**. Un
> `if genero == "isekai": return "fantasia"` enterrado en una función no se le puede
> mostrar a nadie.

In [22]:
mapeo = pd.DataFrame({
    "origen": ["sci-fi", "science fiction", "role-playing", "fantasy", "action"],
    "tag":    ["ciencia-ficcion", "ciencia-ficcion", "rol", "rol", "accion"],
    "nivel":  ["general", "general", "general", "especifico", "general"],
})
ver("La tabla de equivalencias", mapeo)

dic = dict(zip(mapeo["origen"], mapeo["tag"]))      # dos columnas -> diccionario
ver("Como diccionario", dic)

# --- A) map: rápido, UN destino por origen, NO puede duplicar filas ---
largo2 = largo.copy()
largo2["tag"] = largo2["genero"].map(dic)           # lo no encontrado queda NaN
ver("Resultado con map", largo2[["nombre", "genero", "tag"]])

# --- B) merge: trae varias columnas y PUEDE duplicar filas (a veces a propósito) ---
con_tag = largo2.drop(columns="tag").merge(mapeo, left_on="genero",
                                           right_on="origen", how="left")
ver("Resultado con merge (trae también 'nivel')",
    con_tag[["nombre", "genero", "tag", "nivel"]])
ver("¿El merge cambió el número de filas?", {"antes": len(largo2), "después": len(con_tag)})

# --- EL CONTROL DE CALIDAD ---
ver("Géneros SIN equivalencia", con_tag.loc[con_tag["tag"].isna(), "genero"].value_counts())
ver("Cobertura de la tabla", f"{con_tag['tag'].notna().mean():.0%}")

# La tabla se guarda y se recarga como CSV: eso es lo que la hace editable
mapeo.to_csv(TMP / "mapeo.csv", index=False)
ver("Recargada desde CSV como dict",
    pd.read_csv(TMP / "mapeo.csv").set_index("origen")["tag"].to_dict())
ver("Inverso: tag -> lista de orígenes", mapeo.groupby("tag")["origen"].agg(list))
print("""
  Esa tabla de "sin equivalencia" es lo más valioso de la sección. Un ETL que
  mapea en silencio y descarta lo que no reconoce acumula basura durante
  semanas. Uno que REPORTA te obliga a decidir cada caso.

  map vs merge, para elegir con criterio:
    map   -> una columna de destino, no puede duplicar filas. Por defecto.
    merge -> varias columnas; si la tabla derecha tiene claves REPETIDAS
             multiplica filas sin avisar. Verifica len() antes y después.""")


▸ La tabla de equivalencias


,origen,tag,nivel
0,sci-fi,ciencia-ficcion,general
1,science fiction,ciencia-ficcion,general
2,role-playing,rol,general
3,fantasy,rol,especifico
4,action,accion,general



▸ Como diccionario
{'sci-fi': 'ciencia-ficcion', 'science fiction': 'ciencia-ficcion', 'role-playing': 'rol', 'fantasy': 'rol', 'action': 'accion'}

▸ Resultado con map


,nombre,genero,tag
0,Alfa,action,accion
0,Alfa,fantasy,rol
1,Beta,action,accion
2,Gamma,sci-fi,ciencia-ficcion
2,Gamma,drama,NaN
2,Gamma,action,accion
5,Epsilon,fantasy,rol



▸ Resultado con merge (trae también 'nivel')


...


▸ ¿El merge cambió el número de filas?
{'antes': 7, 'después': 7}

▸ Géneros SIN equivalencia


genero
drama    1
Name: count, dtype: int64


▸ Cobertura de la tabla
86%

▸ Recargada desde CSV como dict
{'sci-fi': 'ciencia-ficcion', 'science fiction': 'ciencia-ficcion', 'role-playing': 'rol', 'fantasy': 'rol', 'action': 'accion'}

▸ Inverso: tag -> lista de orígenes


tag
accion                              [action]
ciencia-ficcion    [sci-fi, science fiction]
rol                  [role-playing, fantasy]
Name: origen, dtype: object


  Esa tabla de "sin equivalencia" es lo más valioso de la sección. Un ETL que
  mapea en silencio y descarta lo que no reconoce acumula basura durante
  semanas. Uno que REPORTA te obliga a decidir cada caso.

  map vs merge, para elegir con criterio:
    map   -> una columna de destino, no puede duplicar filas. Por defecto.
    merge -> varias columnas; si la tabla derecha tiene claves REPETIDAS
             multiplica filas sin avisar. Verifica len() antes y después.


---

<a id="sec-14"></a>
## 14. Agrupar y agregar (`groupby`)
<sub>[↑ índice](#indice)</sub>

`groupby` implementa *split–apply–combine*: parte por clave, aplica una función a cada
grupo, recombina. Es la operación que convierte una tabla de títulos en una tabla de
**stands**.

In [23]:
v = pd.DataFrame({
    "empresa": ["acme", "acme", "zeta", "zeta", "zeta", "omega"],
    "tag":     ["accion", "rol", "accion", "terror", "accion", "rol"],
    "ventas":  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
})
ver("Datos base", v)

ver("size(): filas por grupo", v.groupby("empresa").size())
ver("sum() de una columna", v.groupby("empresa")["ventas"].sum())
ver("Varias agregaciones a la vez", v.groupby("empresa")["ventas"].agg(["sum", "mean", "count"]))
ver("nunique(): distintos por grupo", v.groupby("empresa")["tag"].nunique())

# AGREGACIÓN NOMBRADA: nombre_salida=(columna_origen, función). La forma más legible.
res = (v.groupby("empresa")
         .agg(n_filas=("tag", "size"),
              total=("ventas", "sum"),
              # set() evita etiquetas repetidas dentro del mismo stand;
              # sorted() hace el resultado DETERMINISTA entre ejecuciones
              tags=("tag", lambda s: sorted(set(s))))
         .reset_index())                                  # el grupo vuelve a ser columna
ver("Tabla final de stands", res)

ver("Tabla cruzada con groupby + unstack",
    v.groupby(["empresa", "tag"]).size().unstack(fill_value=0))
ver("Lo mismo, más directo, con crosstab", pd.crosstab(v["empresa"], v["tag"]))
ver("pivot_table: el equivalente de la tabla dinámica",
    v.pivot_table(index="empresa", columns="tag", values="ventas", aggfunc="sum", fill_value=0))


▸ Datos base


,empresa,tag,ventas
0,acme,accion,1.0
1,acme,rol,2.0
2,zeta,accion,3.0
3,zeta,terror,4.0
4,zeta,accion,5.0
5,omega,rol,6.0



▸ size(): filas por grupo


empresa
acme     2
omega    1
zeta     3
dtype: int64


▸ sum() de una columna


...

tag,accion,rol,terror
empresa,,,
acme,1,1,0
omega,0,1,0
zeta,2,0,1



▸ Lo mismo, más directo, con crosstab


tag,accion,rol,terror
empresa,,,
acme,1,1,0
omega,0,1,0
zeta,2,0,1



▸ pivot_table: el equivalente de la tabla dinámica


tag,accion,rol,terror
empresa,,,
acme,1.0,2.0,0.0
omega,0.0,6.0,0.0
zeta,8.0,0.0,4.0


In [24]:
# transform: devuelve una fila POR FILA (no por grupo). Sirve para añadir columnas.
v["total_grupo"] = v.groupby("empresa")["ventas"].transform("sum")
ver("transform: el total del grupo pegado a cada fila", v)

v["frec_tag"] = v["tag"].map(v["tag"].value_counts())      # frecuencia GLOBAL por fila
ver("Frecuencia global de cada etiqueta", v[["tag", "frec_tag"]])

v["rank_en_grupo"] = v.groupby("empresa")["ventas"].rank(ascending=False)
ver("Ranking DENTRO de cada grupo", v[["empresa", "ventas", "rank_en_grupo"]])

ver("head(1) por grupo: la primera fila de cada uno", v.groupby("empresa").head(1))
ver("filter: quedarse solo con grupos que cumplen una condición",
    v.groupby("empresa").filter(lambda g: len(g) >= 2))
print("""
  ⚠️ agg vs transform, la confusión clásica:
     agg       -> una fila POR GRUPO   (3 empresas -> 3 filas)
     transform -> una fila POR FILA    (6 filas -> 6 filas, con el valor del grupo)
  Si una agregación te devolvió menos filas de las que esperabas, querías
  transform.

  ⚠️ groupby DESCARTA los nulos de la clave por defecto. Si tus filas sin empresa
  importan, usa dropna=False o te desaparecen sin aviso.

  ⚠️ Olvidar reset_index() deja la clave como ÍNDICE, y el merge posterior falla
  con un KeyError que no menciona el groupby por ninguna parte.""")


▸ transform: el total del grupo pegado a cada fila


,empresa,tag,ventas,total_grupo
0,acme,accion,1.0,3.0
1,acme,rol,2.0,3.0
2,zeta,accion,3.0,12.0
3,zeta,terror,4.0,12.0
4,zeta,accion,5.0,12.0
5,omega,rol,6.0,6.0



▸ Frecuencia global de cada etiqueta


,tag,frec_tag
0,accion,3
1,rol,2
2,accion,3
3,terror,1
4,accion,3
5,rol,2



▸ Ranking DENTRO de cada grupo


...


▸ head(1) por grupo: la primera fila de cada uno


,empresa,tag,ventas,total_grupo,frec_tag,rank_en_grupo
0,acme,accion,1.0,3.0,3,2.0
2,zeta,accion,3.0,12.0,3,3.0
5,omega,rol,6.0,6.0,2,1.0



▸ filter: quedarse solo con grupos que cumplen una condición


,empresa,tag,ventas,total_grupo,frec_tag,rank_en_grupo
0,acme,accion,1.0,3.0,3,2.0
1,acme,rol,2.0,3.0,2,1.0
2,zeta,accion,3.0,12.0,3,3.0
3,zeta,terror,4.0,12.0,1,2.0
4,zeta,accion,5.0,12.0,3,1.0



  ⚠️ agg vs transform, la confusión clásica:
     agg       -> una fila POR GRUPO   (3 empresas -> 3 filas)
     transform -> una fila POR FILA    (6 filas -> 6 filas, con el valor del grupo)
  Si una agregación te devolvió menos filas de las que esperabas, querías
  transform.

  ⚠️ groupby DESCARTA los nulos de la clave por defecto. Si tus filas sin empresa
  importan, usa dropna=False o te desaparecen sin aviso.

  ⚠️ Olvidar reset_index() deja la clave como ÍNDICE, y el merge posterior falla
  con un KeyError que no menciona el groupby por ninguna parte.


---

<a id="sec-15"></a>
## 15. Unir tablas: `merge`, `join`, `concat`
<sub>[↑ índice](#indice)</sub>

> ⚠️ **El error que más cuesta encontrar:** un `merge` que **multiplica filas**. Si la
> tabla derecha tiene la clave repetida, cada fila de la izquierda se duplica por cada
> coincidencia — sin error, sin aviso. `validate=` es el seguro, y casi nadie lo usa.

In [25]:
a = pd.DataFrame({"k": [1, 2, 3], "x": ["a", "b", "c"]})
b = pd.DataFrame({"k": [2, 3, 4], "y": ["B", "C", "D"]})
ver("Tabla a", a); ver("Tabla b", b)

ver("inner: solo las claves en AMBAS", a.merge(b, on="k", how="inner"))
ver("left: todo lo de a; lo que falta queda NaN", a.merge(b, on="k", how="left"))
ver("outer + indicator: de dónde vino cada fila", a.merge(b, on="k", how="outer", indicator=True))
ver("suffixes: cuando hay columnas con el mismo nombre",
    a.merge(b, left_on="k", right_on="k", suffixes=("_a", "_b")))
ver("join: merge por ÍNDICE", a.set_index("k").join(b.set_index("k"), how="left"))
ver("anti-join: filas de a que NO están en b", a[~a["k"].isin(b["k"])])

# validate= convierte un bug silencioso en una excepción clara
try:
    b_dup = pd.DataFrame({"k": [2, 2, 3], "y": ["B1", "B2", "C"]})   # clave 2 REPETIDA
    a.merge(b_dup, on="k", how="left", validate="one_to_one")
except Exception as e:
    ver("validate='one_to_one' detecta el problema", f"{type(e).__name__}: {e}")
ver("Sin validate, el merge duplica en silencio", a.merge(b_dup, on="k", how="left"))
ver("Control manual", {"filas antes": len(a), "filas después": len(a.merge(b_dup, on='k', how='left'))})

# concat: apilar tablas con las mismas columnas (los stands de los 3 dominios)
s1 = pd.DataFrame({"nombre": ["A"], "dominio": ["juego"]})
s2 = pd.DataFrame({"nombre": ["B"], "dominio": ["anime"]})
ver("concat: apilar y reindexar", pd.concat([s1, s2], ignore_index=True))
ver("concat con keys: recuerda de qué tabla vino", pd.concat([s1, s2], keys=["f1", "f2"]))
ver("⚠️ concat con columnas DISTINTAS genera NaN",
    pd.concat([s1, s2.rename(columns={"nombre": "n"})]))
ver("concat axis=1: lado a lado por índice", pd.concat([a, b], axis=1))
print("""
  validate= acepta "one_to_one", "one_to_many", "many_to_one", "many_to_many".
  Ponlo en tu ETL y los problemas de cardinalidad aparecen el día uno, no en la
  presentación cuando alguien pregunte por qué hay 23.000 filas donde debería
  haber 16.598.""")


▸ Tabla a


,k,x
0,1,a
1,2,b
2,3,c



▸ Tabla b


,k,y
0,2,B
1,3,C
2,4,D



▸ inner: solo las claves en AMBAS


...


▸ ⚠️ concat con columnas DISTINTAS genera NaN


,nombre,dominio,n
0,A,juego,NaN
0,NaN,anime,B



▸ concat axis=1: lado a lado por índice


,k,x,k,y
0,1,a,2,B
1,2,b,3,C
2,3,c,4,D



  validate= acepta "one_to_one", "one_to_many", "many_to_one", "many_to_many".
  Ponlo en tu ETL y los problemas de cardinalidad aparecen el día uno, no en la
  presentación cuando alguien pregunte por qué hay 23.000 filas donde debería
  haber 16.598.


---

<a id="sec-16"></a>
## 16. Reestructurar: `melt`, `pivot`, `stack`
<sub>[↑ índice](#indice)</sub>

`melt` y `pivot` son inversos: ancho → largo y largo → ancho. La forma **larga** es la
natural para agrupar y contar; la **ancha**, para mostrar y para alimentar un modelo.

In [26]:
ancho = pd.DataFrame({"id": [1, 2], "accion": [1, 0], "rol": [1, 1]})
ver("Forma ANCHA: una columna por etiqueta", ancho)

largo_m = ancho.melt(id_vars="id",          # columnas que se CONSERVAN como identificador
                     var_name="tag",        # nombre para la columna de nombres antiguos
                     value_name="tiene")    # nombre para la columna de valores
ver("melt: ancho -> largo", largo_m)
ver("pivot: largo -> ancho (el inverso)", largo_m.pivot(index="id", columns="tag", values="tiene"))
ver("Quedarse solo con los 1 (la forma larga útil)", largo_m.query("tiene == 1"))
ver("stack: similar a melt pero deja un índice jerárquico", ancho.set_index("id").stack())
print("""
  Cuándo cada una:
    larga -> groupby, value_counts, merges, cualquier conteo. Es la forma
             "tidy" y casi todo pandas está pensado para ella.
    ancha -> matrices de features para un modelo (NB-11), tablas para mostrar.""")


▸ Forma ANCHA: una columna por etiqueta


,id,accion,rol
0,1,1,1
1,2,0,1



▸ melt: ancho -> largo


,id,tag,tiene
0,1,accion,1
1,2,accion,0
2,1,rol,1
3,2,rol,1



▸ pivot: largo -> ancho (el inverso)


...


▸ Quedarse solo con los 1 (la forma larga útil)


,id,tag,tiene
0,1,accion,1
2,1,rol,1
3,2,rol,1



▸ stack: similar a melt pero deja un índice jerárquico


id        
1   accion    1
    rol       1
2   accion    0
    rol       1
dtype: int64


  Cuándo cada una:
    larga -> groupby, value_counts, merges, cualquier conteo. Es la forma
             "tidy" y casi todo pandas está pensado para ella.
    ancha -> matrices de features para un modelo (NB-11), tablas para mostrar.


---

<a id="sec-17"></a>
## 17. Ordenar, ranking y top-N
<sub>[↑ índice](#indice)</sub>

In [27]:
r = pd.DataFrame({"n": ["a", "b", "c", "d"], "p": [3, 1, 3, 2], "q": [9, 8, 7, 6]})
ver("Datos", r)

ver("sort_values por una columna", r.sort_values("p", ascending=False))
ver("Varios criterios con direcciones distintas", r.sort_values(["p", "q"], ascending=[False, True]))
ver("key=: ordenar aplicando una transformación", r.sort_values("n", key=lambda s: s.str.lower()))
ver("sort_index: ordenar por el índice", r.sort_index())

r["rank_min"] = r["p"].rank(ascending=False, method="min")        # empate -> ambos el menor
r["rank_dense"] = r["p"].rank(ascending=False, method="dense")    # sin huecos tras el empate
ver("Dos formas de rankear con empates", r)

ver("nlargest: top-N (más rápido que ordenar todo)", r.nlargest(2, "p"))
ver("nsmallest: bottom-N", r.nsmallest(2, "q"))
ver("reset_index tras ordenar", r.sort_values("p", ascending=False).reset_index(drop=True))
print("""
  method= en rank() cambia qué pasa con los empates:
    'min'     1,2,2,4   deja hueco (es el ranking deportivo)
    'dense'   1,2,2,3   sin huecos
    'first'   1,2,3,4   desempata por orden de aparición
    'average' 1,2.5,2.5,4
  El ranking de stands que le muestres al asistente depende de esta elección.""")


▸ Datos


,n,p,q
0,a,3,9
1,b,1,8
2,c,3,7
3,d,2,6



▸ sort_values por una columna


,n,p,q
0,a,3,9
2,c,3,7
3,d,2,6
1,b,1,8



▸ Varios criterios con direcciones distintas


...


▸ nsmallest: bottom-N


,n,p,q,rank_min,rank_dense
3,d,2,6,3.0,2.0
2,c,3,7,1.0,1.0



▸ reset_index tras ordenar


,n,p,q,rank_min,rank_dense
0,a,3,9,1.0,1.0
1,c,3,7,1.0,1.0
2,d,2,6,3.0,2.0
3,b,1,8,4.0,3.0



  method= en rank() cambia qué pasa con los empates:
    'min'     1,2,2,4   deja hueco (es el ranking deportivo)
    'dense'   1,2,2,3   sin huecos
    'first'   1,2,3,4   desempata por orden de aparición
    'average' 1,2.5,2.5,4
  El ranking de stands que le muestres al asistente depende de esta elección.


---

<a id="sec-18"></a>
## 18. Ventanas y acumulados
<sub>[↑ índice](#indice)</sub>

Operaciones donde el valor de una fila depende de las **vecinas**. Son la base del
análisis de series de tiempo: tráfico por hora de la feria, mediciones de la tesis.

In [28]:
w = pd.DataFrame({"dia": pd.date_range("2026-11-01", periods=5), "v": [1, 3, 2, 5, 4]})

w["acum"]   = w["v"].cumsum()              # suma acumulada (cumprod, cummax, cummin)
w["media3"] = w["v"].rolling(3).mean()     # media móvil de 3: las 2 primeras son NaN
w["dif"]    = w["v"].diff()                # diferencia con la fila anterior
w["ant"]    = w["v"].shift(1)              # desplazar una posición hacia abajo
w["pct"]    = w["v"].pct_change()          # variación relativa respecto de la anterior
ver("Todas las ventanas juntas", w)

ver("resample: reagrupar por periodos de tiempo",
    w.set_index("dia").resample("2D")["v"].sum())
print("""
  ⚠️ rolling(3) produce NaN en las 2 primeras filas: no hay 3 valores todavía.
  No es un error, y si lo "arreglas" con fillna(0) estás inventando datos.
  Si necesitas un valor desde el principio, usa min_periods=1 y asume que las
  primeras medias están calculadas sobre menos puntos.

  ⚠️ shift() es lo que usas para construir features "del pasado" en un modelo.
  Si por error desplazas al revés (shift(-1)) metes información del FUTURO y el
  modelo da resultados espectaculares que no se reproducen. Es fuga de datos.""")


▸ Todas las ventanas juntas


,dia,v,acum,media3,dif,ant,pct
0,2026-11-01,1,1,NaN,NaN,NaN,NaN
1,2026-11-02,3,4,NaN,2.0,1.0,2.000000
2,2026-11-03,2,6,2.000000,-1.0,3.0,-0.333333
3,2026-11-04,5,11,3.333333,3.0,2.0,1.500000
4,2026-11-05,4,15,3.666667,-1.0,5.0,-0.200000



▸ resample: reagrupar por periodos de tiempo


dia
2026-11-01    4
2026-11-03    7
2026-11-05    4
Freq: 2D, Name: v, dtype: int64


  ⚠️ rolling(3) produce NaN en las 2 primeras filas: no hay 3 valores todavía.
  No es un error, y si lo "arreglas" con fillna(0) estás inventando datos.
  Si necesitas un valor desde el principio, usa min_periods=1 y asume que las
  primeras medias están calculadas sobre menos puntos.

  ⚠️ shift() es lo que usas para construir features "del pasado" en un modelo.
  Si por error desplazas al revés (shift(-1)) metes información del FUTURO y el
  modelo da resultados espectaculares que no se reproducen. Es fuga de datos.


---

<a id="parte-d"></a>
# Parte D · Matemática y similitud

<a id="sec-19"></a>
## 19. NumPy: lo que de verdad necesitas
<sub>[↑ índice](#indice)</sub>

pandas está construido **sobre** NumPy: cada columna numérica es, por debajo, un
`ndarray`. Entender NumPy explica por qué pandas es rápido y por qué un bucle de Python
sobre un DataFrame es lento.

```
LISTA DE PYTHON                    ARRAY DE NUMPY
[1, 2, 3]                          array([1, 2, 3])
 ↓ cada elemento es un OBJETO       ↓ un solo bloque contiguo de memoria
 punteros dispersos en memoria      tipo único y fijo (int64)
 el bucle lo hace el intérprete     el bucle lo hace C, sin intérprete
```

Esa diferencia es la que produce los factores de 50–100× que medimos en §23.

In [29]:
import numpy as np, pandas as pd
from IPython.display import display

a = np.array([1, 2, 3, 4], dtype=float)      # dtype explícito: evita sorpresas de tipo
M = np.array([[1, 2], [3, 4]], dtype=float)  # matriz 2×2

ver("Vector", a)
ver("Matriz", M)
ver("shape / dtype / ndim", (a.shape, a.dtype, a.ndim))

ver("zeros / ones / arange / linspace",
    (np.zeros(3), np.ones(2), np.arange(0, 5, 2), np.linspace(0, 1, 3)))
ver("eye: matriz identidad", np.eye(2))

# Vectorización: la operación se aplica a TODO el array sin bucle de Python
ver("a * 2 (sin bucle)", a * 2)
ver("a + a", a + a)
ver("a ** 2", a ** 2)
ver("np.sqrt(a)", np.sqrt(a))
ver("np.log1p(a): log(1+x), estable cuando x es pequeño", np.log1p(a))

ver("sum / mean / std / min / max",
    (a.sum(), a.mean(), round(float(a.std()), 4), a.min(), a.max()))
ver("axis=0 suma por COLUMNAS", M.sum(axis=0))
ver("axis=1 suma por FILAS", M.sum(axis=1))
ver("argmax / argmin: la POSICIÓN del extremo, no el valor", (a.argmax(), a.argmin()))
ver("argsort: las posiciones que ordenarían el array", np.argsort(-a))

ver("M @ M: producto matricial", M @ M)
ver("M * M: producto ELEMENTO A ELEMENTO (distinto de @)", M * M)
ver("norma euclídea ||a||", float(np.linalg.norm(a)))
ver("producto punto a·a", float(a @ a))
print("""
  ⚠️ @ vs *: @ es el producto matricial (filas por columnas); * multiplica
  posición por posición. Confundirlos no lanza error si las formas calzan, y
  da un resultado numérico equivocado. En NB-11 esa confusión arruina la
  ecuación normal.

  ⚠️ axis= se lee así: axis=0 "colapsa las FILAS" (resultado por columna),
  axis=1 "colapsa las COLUMNAS" (resultado por fila). La mnemotecnia: el eje
  que nombras es el que DESAPARECE.""")


▸ Vector
[1. 2. 3. 4.]

▸ Matriz
[[1. 2.]
 [3. 4.]]

▸ shape / dtype / ndim
((4,), dtype('float64'), 1)

▸ zeros / ones / arange / linspace
(array([0., 0., 0.]), array([1., 1.]), array([0, 2, 4]), array([0. , 0.5, 1. ]))

▸ eye: matriz identidad
[[1. 0.]
 [0. 1.]]

▸ a * 2 (sin bucle)
[2. 4. 6. 8.]

▸ a + a
[2. 4. 6. 8.]

▸ a ** 2
[ 1.  4.  9. 16.]

▸ np.sqrt(a)
[1.         1.41421356 1.73205081 2.        ]

▸ np.log1p(a): log(1+x), estable cuando x es pequeño
[0.69314718 1.09861229 1.38629436 1.60943791]

▸ sum / mean / std / min / max
(np.float64(10.0), np.float64(2.5), 1.118, np.float64(1.0), np.float64(4.0))

▸ axis=0 suma por COLUMNAS
[4. 6.]

▸ axis=1 suma por FILAS
[3. 7.]

▸ argmax / argmin: la POSICIÓN del extremo, no el valor
(np.int64(3), np.int64(0))

▸ argsort: las posiciones que ordenarían el array
[3 2 1 0]

▸ M @ M: producto matricial
[[ 7. 10.]
 [15. 22.]]

▸ M * M: producto ELEMENTO A ELEMENTO (distinto de @)
[[ 1.  4.]
 [ 9. 16.]]

▸ norma euclídea ||a||
5.477225575

In [30]:
# --- Broadcasting: operar arrays de formas distintas sin copiarlos ---
fila = np.array([[1, 2, 3]])        # forma (1,3)
col  = np.array([[10], [20]])       # forma (2,1)
ver("fila + col -> (2,3) por broadcasting", fila + col)
print("""
  La regla: NumPy compara las formas de derecha a izquierda; dos dimensiones son
  compatibles si son iguales o si una de ellas es 1, y la de tamaño 1 se
  "estira" virtualmente (sin copiar memoria).
    (2,1) + (1,3) -> (2,3)   ✓
    (2,3) + (3,)  -> (2,3)   ✓  (se alinea por la derecha)
    (2,3) + (2,)  -> ERROR      (3 vs 2 por la derecha)
  Si te da un error de shapes, mira las formas alineadas a la DERECHA.""")

# --- Normalizar filas: la operación previa a cualquier coseno ---
X = np.array([[3.0, 4.0], [1.0, 0.0]])
normas = np.linalg.norm(X, axis=1, keepdims=True)   # keepdims mantiene (2,1) en vez de (2,)
ver("Normas por fila (keepdims=True da forma (2,1))", normas)
ver("Filas normalizadas a norma 1", X / normas)     # broadcasting (2,2)/(2,1)
ver("Verificación: todas las normas valen 1", np.linalg.norm(X / normas, axis=1))
print("""
  ⚠️ keepdims=True no es cosmético: sin él la norma tiene forma (2,) y
  X / normas aplica broadcasting por COLUMNAS, dividiendo mal. El resultado
  sale sin error y está equivocado — el peor tipo de bug.""")

# --- Máscaras y where en NumPy ---
ver("Máscara booleana", a > 2)
ver("Selección por máscara", a[a > 2])
ver("np.where como if/else vectorizado", np.where(a > 2, "alto", "bajo"))
ver("np.clip: recortar a un rango", np.clip(a, 2, 3))
ver("np.isnan detecta NaN (x == np.nan NO funciona)", np.isnan(np.array([1.0, np.nan])))


▸ fila + col -> (2,3) por broadcasting
[[11 12 13]
 [21 22 23]]

  La regla: NumPy compara las formas de derecha a izquierda; dos dimensiones son
  compatibles si son iguales o si una de ellas es 1, y la de tamaño 1 se
  "estira" virtualmente (sin copiar memoria).
    (2,1) + (1,3) -> (2,3)   ✓
    (2,3) + (3,)  -> (2,3)   ✓  (se alinea por la derecha)
    (2,3) + (2,)  -> ERROR      (3 vs 2 por la derecha)
  Si te da un error de shapes, mira las formas alineadas a la DERECHA.

▸ Normas por fila (keepdims=True da forma (2,1))
[[5.]
 [1.]]

▸ Filas normalizadas a norma 1
[[0.6 0.8]
 [1.  0. ]]

▸ Verificación: todas las normas valen 1
[1. 1.]

  ⚠️ keepdims=True no es cosmético: sin él la norma tiene forma (2,) y
  X / normas aplica broadcasting por COLUMNAS, dividiendo mal. El resultado
  sale sin error y está equivocado — el peor tipo de bug.

▸ Máscara booleana
[False False  True  True]

▸ Selección por máscara
[3. 4.]

▸ np.where como if/else vectorizado
['bajo' 'bajo' 'alto' 'alto

---

<a id="sec-20"></a>
## 20. Datos aleatorios y sintéticos
<sub>[↑ índice](#indice)</sub>

> **La regla de reproducibilidad.** `np.random.default_rng(SEMILLA)` en vez de
> `np.random.seed()`. El generador moderno está **encapsulado**: cada `rng` lleva su
> propio estado, así que una función que recibe un `rng` no altera el azar del resto del
> programa. `np.random.seed()` modifica un estado **global** y cualquier librería que
> llame a `np.random` por detrás te cambia los resultados.
>
> Sin semilla no puedes reproducir un resultado, y si no puedes reproducirlo no puedes
> depurarlo ni defenderlo.

In [31]:
rng = np.random.default_rng(42)               # 42 = semilla; cualquier entero sirve

ver("uniform: continua entre a y b", rng.uniform(0, 1, 3).round(3))
ver("integers: enteros (el extremo superior NO se incluye)", rng.integers(1, 7, 5))
ver("normal: campana con media y desviación", rng.normal(100, 15, 4).round(1))
ver("choice: elegir de una lista", rng.choice(["a", "b", "c"], 5))
ver("choice con PESOS: 'a' 7 veces más probable",
    rng.choice(["a", "b"], 10, p=[0.875, 0.125]))
ver("choice sin reemplazo: no repite", rng.choice(np.arange(10), 4, replace=False))
ver("permutation: mezclar", rng.permutation(np.arange(5)))

# Reproducibilidad: dos generadores con la MISMA semilla dan lo MISMO
ver("Dos rng con semilla 7 dan lo mismo",
    (np.random.default_rng(7).integers(0, 100, 3),
     np.random.default_rng(7).integers(0, 100, 3)))


▸ uniform: continua entre a y b
[0.774 0.439 0.859]

▸ integers: enteros (el extremo superior NO se incluye)
[1 5 2 1 4]

▸ normal: campana con media y desviación
[101.9  95.3  99.7  87.2]

▸ choice: elegir de una lista
['c' 'b' 'b' 'a' 'c']

▸ choice con PESOS: 'a' 7 veces más probable
['a' 'a' 'a' 'a' 'a' 'a' 'a' 'a' 'a' 'a']

▸ choice sin reemplazo: no repite
[0 7 8 4]

▸ permutation: mezclar
[0 2 1 3 4]

▸ Dos rng con semilla 7 dan lo mismo
(array([94, 62, 68]), array([94, 62, 68]))


In [32]:
# --- Generar un dataset sintético completo y coherente ---
SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)
N = 200

# Distribución LEY DE POTENCIA: pocos productores con muchas obras, muchos con pocas.
# Es la forma real de catálogos, ventas y popularidad; la uniforme no lo es.
rangos = np.arange(1, 21)                       # 20 productores, rango 1..20
pesos_pot = 1.0 / rangos                        # peso ∝ 1/rango  (Zipf con exponente 1)
pesos_pot = pesos_pot / pesos_pot.sum()         # normalizar para que sumen 1

sint = pd.DataFrame({
    "id": np.arange(1, N + 1),
    "productor": rng.choice([f"P{i:02d}" for i in rangos], N, p=pesos_pot),
    "anio": rng.integers(1995, 2027, N),
    # lognormal: positiva y con cola larga a la derecha, como los precios reales
    "precio": np.round(rng.lognormal(mean=2.5, sigma=0.6, size=N), 2),
    "tag": rng.choice(["accion", "rol", "terror", "aventura", "simulacion"], N,
                      p=[0.35, 0.25, 0.15, 0.15, 0.10]),
})
ver("Dataset sintético", sint)
ver("Concentración: top-3 productores de 20", sint["productor"].value_counts().head(3))

# La MEDICIÓN que demuestra que la ley de potencia hace diferencia
top3_pot = sint["productor"].value_counts().head(3).sum() / N
unif = pd.Series(np.random.default_rng(SEMILLA).choice(
    [f"P{i:02d}" for i in rangos], N))          # sin p= -> uniforme
top3_unif = unif.value_counts().head(3).sum() / N
ver("Share del top-3", {"ley de potencia": f"{top3_pot:.1%}", "uniforme": f"{top3_unif:.1%}"})
print(f"""
  La diferencia medida ({top3_pot:.0%} vs {top3_unif:.0%}, un factor de
  {top3_pot/top3_unif:.1f}×) no es un detalle estético. Si pruebas tu
  recomendador con datos uniformes, no verás el problema de los stands enormes
  que concentran todas las etiquetas — el problema que SÍ tendrás con datos
  reales. Un dataset sintético mal distribuido valida un sistema que después
  falla.

  Prueba de olor: si en tus datos "reales" los 20 productores tienen todos casi
  el mismo número de obras, sospecha que son sintéticos o que están filtrados.""")


▸ Dataset sintético


,id,productor,anio,precio,tag
0,1,P01,2022,50.22,accion
1,2,P06,2019,21.92,accion
2,3,P03,2018,13.51,rol
3,4,P02,2015,12.61,accion
4,5,P02,1995,15.97,rol
5,6,P10,2001,8.24,aventura
6,7,P15,2022,14.44,rol
7,8,P01,2018,7.00,rol



▸ Concentración: top-3 productores de 20


productor
P01    46
P02    35
P03    19
Name: count, dtype: int64


▸ Share del top-3
{'ley de potencia': '50.0%', 'uniforme': '20.0%'}

  La diferencia medida (50% vs 20%, un factor de
  2.5×) no es un detalle estético. Si pruebas tu
  recomendador con datos uniformes, no verás el problema de los stands enormes
  que concentran todas las etiquetas — el problema que SÍ tendrás con datos
  reales. Un dataset sintético mal distribuido valida un sistema que después
  falla.

  Prueba de olor: si en tus datos "reales" los 20 productores tienen todos casi
  el mismo número de obras, sospecha que son sintéticos o que están filtrados.


---

<a id="sec-21"></a>
## 21. Similitud: Jaccard, IDF y coseno
<sub>[↑ índice](#indice)</sub>

Tres formas de medir "parecido" entre dos conjuntos de etiquetas, y la diferencia
importa:

```
JACCARD           |A ∩ B| / |A ∪ B|        todas las etiquetas valen igual
JACCARD PONDERADO Σ idf(∩) / Σ idf(∪)      las etiquetas RARAS valen más
COSENO            (A·B)/(|A||B|)           sobre vectores; permite pesos y conteos
```

`idf(t) = log(N / df(t))`: si una etiqueta aparece en **todos** los ítems, `df=N`,
`idf=log(1)=0` y deja de aportar. Es lo que hace que compartir `"accion"` (común)
informe menos que compartir `"cyberpunk"` (raro).

In [33]:
import math
from collections import Counter

docs = {
    "A": {"accion", "rol", "fantasia"},
    "B": {"accion", "rol"},
    "C": {"accion", "terror"},
    "D": {"accion", "fantasia", "cyberpunk"},     # cyberpunk aparece UNA vez: idf alto
}
N_docs = len(docs)

def jaccard(x: set, y: set) -> float:
    """|x ∩ y| / |x ∪ y|. Devuelve 0 si ambos están vacíos (evita 0/0)."""
    if not x and not y:
        return 0.0
    return len(x & y) / len(x | y)               # & = intersección, | = unión

ver("Jaccard A-B (2 comunes de 3)", round(jaccard(docs["A"], docs["B"]), 3))
ver("Jaccard A-C (1 común de 4)", round(jaccard(docs["A"], docs["C"]), 3))
ver("Jaccard A-D (2 comunes de 4)", round(jaccard(docs["A"], docs["D"]), 3))

# --- IDF: cuántos documentos contienen cada etiqueta ---
df_tag = Counter(t for s in docs.values() for t in s)      # document frequency
idf = {t: math.log(N_docs / c) for t, c in df_tag.items()} # log(N/df)
ver("document frequency por etiqueta", dict(df_tag))
ver("idf por etiqueta (ordenado)", {k: round(v, 3) for k, v in
                                    sorted(idf.items(), key=lambda kv: -kv[1])})
print("""
  'accion' está en los 4 documentos: idf = log(4/4) = 0. No aporta NADA a la
  similitud, y eso es correcto: que dos obras compartan la etiqueta más común
  del catálogo no dice nada sobre si se parecen.""")


▸ Jaccard A-B (2 comunes de 3)
0.667

▸ Jaccard A-C (1 común de 4)
0.25

▸ Jaccard A-D (2 comunes de 4)
0.5

▸ document frequency por etiqueta
{'fantasia': 2, 'rol': 2, 'accion': 4, 'terror': 1, 'cyberpunk': 1}

▸ idf por etiqueta (ordenado)
{'terror': 1.386, 'cyberpunk': 1.386, 'fantasia': 0.693, 'rol': 0.693, 'accion': 0.0}

  'accion' está en los 4 documentos: idf = log(4/4) = 0. No aporta NADA a la
  similitud, y eso es correcto: que dos obras compartan la etiqueta más común
  del catálogo no dice nada sobre si se parecen.


In [34]:
def jaccard_idf(x: set, y: set, idf: dict) -> float:
    """Jaccard ponderado por idf: Σidf(∩) / Σidf(∪).

    Mismo cociente que Jaccard, pero contando "peso informativo" en vez de
    elementos. Compartir una etiqueta rara sube más que compartir una común.
    """
    inter = sum(idf.get(t, 0.0) for t in x & y)      # peso de lo compartido
    union = sum(idf.get(t, 0.0) for t in x | y)      # peso total en juego
    return inter / union if union > 0 else 0.0

pares = [("A", "B"), ("A", "C"), ("A", "D")]
comp = pd.DataFrame([{
    "par": f"{p}-{q}",
    "comunes": sorted(docs[p] & docs[q]),
    "jaccard": round(jaccard(docs[p], docs[q]), 3),
    "jaccard_idf": round(jaccard_idf(docs[p], docs[q], idf), 3),
} for p, q in pares])
ver("Jaccard simple vs ponderado", comp)
print("""
  Lee la tabla: A-B comparte {accion, rol} y A-D comparte {accion, fantasia}.
  Jaccard simple los trata distinto solo por el tamaño de la unión. El ponderado
  descuenta 'accion' (idf=0) en los dos casos, y lo que queda es la comparación
  entre 'rol' (idf=0.69) y 'fantasia' (idf=0.69) — que son igual de informativas.

  ⚠️ No esperes que el IDF cambie el ranking SIEMPRE. Con 4 documentos y 5
  etiquetas el rango de idf es estrecho. El efecto crece con el catálogo y con
  la variedad de etiquetas: en 16.598 juegos con 12 géneros el idf casi no
  mueve nada; en 20.000 animes con cientos de tags, sí. Medirlo en TUS datos
  es parte del trabajo — no lo asumas.""")


▸ Jaccard simple vs ponderado


,par,comunes,jaccard,jaccard_idf
0,A-B,"[accion, rol]",0.667,0.50
1,A-C,[accion],0.250,0.00
2,A-D,"[accion, fantasia]",0.500,0.25



  Lee la tabla: A-B comparte {accion, rol} y A-D comparte {accion, fantasia}.
  Jaccard simple los trata distinto solo por el tamaño de la unión. El ponderado
  descuenta 'accion' (idf=0) en los dos casos, y lo que queda es la comparación
  entre 'rol' (idf=0.69) y 'fantasia' (idf=0.69) — que son igual de informativas.

  ⚠️ No esperes que el IDF cambie el ranking SIEMPRE. Con 4 documentos y 5
  etiquetas el rango de idf es estrecho. El efecto crece con el catálogo y con
  la variedad de etiquetas: en 16.598 juegos con 12 géneros el idf casi no
  mueve nada; en 20.000 animes con cientos de tags, sí. Medirlo en TUS datos
  es parte del trabajo — no lo asumas.


In [35]:
# --- Coseno sobre la matriz obra × etiqueta: todos los pares de una vez ---
tags = sorted({t for s in docs.values() for t in s})       # vocabulario ordenado
nombres = list(docs)

# Matriz binaria: 1 si la obra tiene la etiqueta. Es la "bag of tags".
B = np.array([[1 if t in docs[n] else 0 for t in tags] for n in nombres], dtype=float)
ver("Matriz obra × etiqueta", pd.DataFrame(B, index=nombres, columns=tags).astype(int))

# Versión PONDERADA: cada columna multiplicada por el idf de su etiqueta
W = B * np.array([idf[t] for t in tags])
ver("Matriz ponderada por idf", pd.DataFrame(W, index=nombres, columns=tags).round(2))

def coseno_matricial(M: np.ndarray) -> np.ndarray:
    """Coseno entre TODAS las filas, de una sola vez.

    cos(i,j) = (fila_i · fila_j) / (||fila_i|| ||fila_j||). Normalizar primero
    las filas y luego hacer Mn @ Mn.T calcula todos los pares con UN producto
    matricial, en vez de un doble bucle de Python.
    """
    normas = np.linalg.norm(M, axis=1, keepdims=True)   # (n,1); keepdims es clave
    normas[normas == 0] = 1.0                            # fila vacía -> evitar 0/0
    Mn = M / normas                                      # cada fila con norma 1
    return Mn @ Mn.T                                     # (n,n) con todos los cosenos

S_bin = coseno_matricial(B)
S_idf = coseno_matricial(W)
ver("Coseno binario", pd.DataFrame(S_bin, index=nombres, columns=nombres).round(3))
ver("Coseno ponderado por idf", pd.DataFrame(S_idf, index=nombres, columns=nombres).round(3))

def top_k_excluyendo(S: np.ndarray, i: int, k: int, nombres: list):
    """Top-k más parecidos a la fila i, SIN incluir la fila i.

    Poner -inf en la posición propia es la forma robusta: con empates en 1.0
    (dos obras con etiquetas idénticas) un simple "saltar el primero" puede
    descartar un vecino legítimo y dejar dentro la obra de referencia.
    """
    puntajes = S[i].copy()
    puntajes[i] = -np.inf
    orden = np.argsort(-puntajes)[:k]
    return [(nombres[j], round(float(puntajes[j]), 3)) for j in orden]

ver("Top-2 parecidos a A (binario)", top_k_excluyendo(S_bin, 0, 2, nombres))
ver("Top-2 parecidos a A (idf)", top_k_excluyendo(S_idf, 0, 2, nombres))
print("""
  La diagonal vale 1.0: toda obra es idéntica a sí misma. Ese es exactamente el
  bug del "recomendador que se recomienda a sí mismo", y por eso la exclusión
  explícita con -inf.

  Coseno vs Jaccard: sobre vectores BINARIOS el coseno es el índice de Ochiai,
  |A∩B|/√(|A||B|), y ordena casi igual que Jaccard. La ventaja del coseno es que
  admite PESOS (idf) y CONTEOS, y que la versión matricial calcula n² pares con
  un solo @ en vez de un doble bucle.""")


▸ Matriz obra × etiqueta


,accion,cyberpunk,fantasia,rol,terror
A,1,0,1,1,0
B,1,0,0,1,0
C,1,0,0,0,1
D,1,1,1,0,0



▸ Matriz ponderada por idf


,accion,cyberpunk,fantasia,rol,terror
A,0.0,0.00,0.69,0.69,0.00
B,0.0,0.00,0.00,0.69,0.00
C,0.0,0.00,0.00,0.00,1.39
D,0.0,1.39,0.69,0.00,0.00



▸ Coseno binario


,A,B,C,D
A,1.000,0.816,0.408,0.667
B,0.816,1.000,0.500,0.408
C,0.408,0.500,1.000,0.408
D,0.667,0.408,0.408,1.000



▸ Coseno ponderado por idf


,A,B,C,D
A,1.000,0.707,0.0,0.316
B,0.707,1.000,0.0,0.000
C,0.000,0.000,1.0,0.000
D,0.316,0.000,0.0,1.000



▸ Top-2 parecidos a A (binario)
[('B', 0.816), ('D', 0.667)]

▸ Top-2 parecidos a A (idf)
[('B', 0.707), ('D', 0.316)]

  La diagonal vale 1.0: toda obra es idéntica a sí misma. Ese es exactamente el
  bug del "recomendador que se recomienda a sí mismo", y por eso la exclusión
  explícita con -inf.

  Coseno vs Jaccard: sobre vectores BINARIOS el coseno es el índice de Ochiai,
  |A∩B|/√(|A||B|), y ordena casi igual que Jaccard. La ventaja del coseno es que
  admite PESOS (idf) y CONTEOS, y que la versión matricial calcula n² pares con
  un solo @ en vez de un doble bucle.


---

<a id="parte-e"></a>
# Parte E · Calidad y rendimiento

<a id="sec-22"></a>
## 22. Validación: hacer que el error se declare
<sub>[↑ índice](#indice)</sub>

> **Idea clave.** Un ETL sin validación no está "funcionando": está **callado**. La
> diferencia entre un pipeline profesional y un script es que el profesional **afirma**
> lo que cree y se detiene cuando deja de ser cierto.

Las tres propiedades que hay que afirmar siempre:

1. **Idempotencia** — correrlo dos veces deja la base igual que correrlo una.
2. **Conservación** — `leídos = cargados + rechazados`. Sin excepción.
3. **Integridad referencial** — toda clave foránea existe en la tabla destino.

In [36]:
val = pd.DataFrame({"id": [1, 2, 3], "tag": ["a", "b", "a"],
                    "precio": [10.0, 20.0, 30.0], "stand": ["s1", "s1", "s2"]})

# Un assert por INVARIANTE, con mensaje que diga qué se rompió y cuánto
assert val["id"].is_unique, "ids duplicados"
assert val["id"].notna().all(), "ids nulos"
assert (val["precio"] > 0).all(), f"precios no positivos: {(val['precio'] <= 0).sum()}"
assert val["tag"].isin(["a", "b", "c"]).all(), \
    f"tags fuera del catálogo: {set(val['tag']) - {'a','b','c'}}"
print("✓ Todas las invariantes se cumplen")

# Integridad referencial: ningún stand huérfano
stands = pd.DataFrame({"stand_id": ["s1", "s2"]})
huerfanos = set(val["stand"]) - set(stands["stand_id"])
assert not huerfanos, f"stands inexistentes: {huerfanos}"
print("✓ Integridad referencial verificada")

# Ver el assert FALLANDO, que es lo que de verdad enseña
try:
    malo = val.assign(precio=[10.0, -5.0, 30.0])
    assert (malo["precio"] > 0).all(), f"precios no positivos: {(malo['precio'] <= 0).sum()}"
except AssertionError as e:
    ver("El assert se dispara con un mensaje útil", str(e))

ver("Rangos observados (lo que un assert debería acotar)",
    val[["precio"]].agg(["min", "max", "mean"]))
print("""
  El mensaje del assert es lo que importa. "assert ok" no te dice nada a las
  2 de la mañana; "precios no positivos: 1" te manda directo a la fila.""")

✓ Todas las invariantes se cumplen
✓ Integridad referencial verificada

▸ El assert se dispara con un mensaje útil
precios no positivos: 1

▸ Rangos observados (lo que un assert debería acotar)


,precio
min,10.0
max,30.0
mean,20.0



  El mensaje del assert es lo que importa. "assert ok" no te dice nada a las
  2 de la mañana; "precios no positivos: 1" te manda directo a la fila.


In [37]:
def validar(d: pd.DataFrame, clave: str, catalogo: set) -> dict:
    """Valida y devuelve un INFORME en vez de solo fallar.

    Un informe sirve para dos cosas que un assert no da: ponerlo en la
    presentación, y decidir si el problema es tolerable o bloqueante.
    """
    return {
        "filas": len(d),
        "clave_unica": bool(d[clave].is_unique),
        "nulos_clave": int(d[clave].isna().sum()),
        "fuera_catalogo": sorted(set(d["tag"].dropna()) - catalogo),
        "pct_nulos_total": round(float(d.isna().mean().mean() * 100), 2),
    }

ver("Informe de validación", validar(val, "id", {"a", "b", "c"}))
ver("Informe con un problema inyectado",
    validar(val.assign(tag=["a", "zzz", "a"]), "id", {"a", "b", "c"}))
print("""
🎯 Para tu proyecto — los asserts que te van a salvar:
     assert obras["obra_id"].is_unique
     assert set(obras["stand_id"]) <= set(stands["stand_id"])
     assert etiquetas["tag"].isin(catalogo_oficial).all()
     assert leidos == cargados + rechazados
   El cuarto es el que te permite decir en la defensa, con un número, por qué
   entraron 14.203 de 16.598 registros. Sin él, la respuesta es "se perdieron
   algunos", y eso cuesta nota.""")


▸ Informe de validación
{'filas': 3, 'clave_unica': True, 'nulos_clave': 0, 'fuera_catalogo': [], 'pct_nulos_total': 0.0}

▸ Informe con un problema inyectado
{'filas': 3, 'clave_unica': True, 'nulos_clave': 0, 'fuera_catalogo': ['zzz'], 'pct_nulos_total': 0.0}

🎯 Para tu proyecto — los asserts que te van a salvar:
     assert obras["obra_id"].is_unique
     assert set(obras["stand_id"]) <= set(stands["stand_id"])
     assert etiquetas["tag"].isin(catalogo_oficial).all()
     assert leidos == cargados + rechazados
   El cuarto es el que te permite decir en la defensa, con un número, por qué
   entraron 14.203 de 16.598 registros. Sin él, la respuesta es "se perdieron
   algunos", y eso cuesta nota.


---

<a id="sec-23"></a>
## 23. Rendimiento: qué cuesta caro
<sub>[↑ índice](#indice)</sub>

La jerarquía, de más rápido a más lento:

```
1. operación vectorizada de pandas/NumPy   ← el intérprete no entra en el bucle
2. .map con un dict                        ← una búsqueda hash por elemento
3. .apply(axis=0)                           ← Python por columna
4. .apply(axis=1)                           ← Python por FILA
5. for fila in df.iterrows()                ← además construye una Series por fila
```

In [38]:
import time

grande = pd.DataFrame({"a": np.arange(200_000), "b": np.arange(200_000) * 2})

t0 = time.perf_counter()
_ = grande["a"] + grande["b"]                               # vectorizado
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
_ = grande.apply(lambda f: f["a"] + f["b"], axis=1)         # fila a fila
t_apply = time.perf_counter() - t0

ver("Tiempos", {"vectorizado": f"{t_vec*1000:.2f} ms",
                "apply(axis=1)": f"{t_apply*1000:.0f} ms",
                "factor": f"{t_apply/max(t_vec,1e-9):.0f}×"})

# iterrows sobre una muestra pequeña: con las 200.000 filas tardaría demasiado
t0 = time.perf_counter()
_ = [f["a"] + f["b"] for _, f in grande.head(5_000).iterrows()]
t_iter = time.perf_counter() - t0
ver("iterrows extrapolado a 200.000 filas", f"≈ {t_iter * 40 * 1000:.0f} ms")
print(f"""
  ⚠️ El número exacto varía con la máquina, pero el ORDEN DE MAGNITUD no: aquí
  salieron {t_apply/max(t_vec,1e-9):.0f}×, es decir dos o tres órdenes de
  magnitud. Con 200.000 filas eso es la diferencia entre {t_vec*1000:.1f} ms y
  {t_apply*1000:.0f} ms.

  Cuándo apply(axis=1) es aceptable: cuando la lógica es irreductiblemente por
  fila Y el DataFrame es pequeño (cientos de filas). Para todo lo demás hay una
  versión vectorizada: np.where, np.select, map con dict, operaciones de
  columnas, o un merge.""")


▸ Tiempos
{'vectorizado': '1.51 ms', 'apply(axis=1)': '1026 ms', 'factor': '681×'}

▸ iterrows extrapolado a 200.000 filas
≈ 3101 ms

  ⚠️ El número exacto varía con la máquina, pero el ORDEN DE MAGNITUD no: aquí
  salieron 681×, es decir dos o tres órdenes de
  magnitud. Con 200.000 filas eso es la diferencia entre 1.5 ms y
  1026 ms.

  Cuándo apply(axis=1) es aceptable: cuando la lógica es irreductiblemente por
  fila Y el DataFrame es pequeño (cientos de filas). Para todo lo demás hay una
  versión vectorizada: np.where, np.select, map con dict, operaciones de
  columnas, o un merge.


In [39]:
# --- Memoria: medirla antes de optimizarla ---
m = pd.DataFrame({"txt": ["categoria_" + str(i % 5) for i in range(50_000)],
                  "num": np.arange(50_000)})
antes = m.memory_usage(deep=True).sum() / 1e6     # deep=True mide el contenido de los strings
m["txt"] = m["txt"].astype("category")            # category guarda códigos + diccionario
despues = m.memory_usage(deep=True).sum() / 1e6
ver("Memoria antes y después de category",
    {"antes": f"{antes:.2f} MB", "después": f"{despues:.2f} MB",
     "ahorro": f"{(1-despues/antes):.0%}"})

ver("downcast de enteros",
    {"int64": pd.Series(np.arange(1000)).memory_usage(deep=True),
     "int16": pd.Series(np.arange(1000), dtype="int16").memory_usage(deep=True)})
print("""
  category gana cuando hay MUCHAS filas y POCOS valores distintos (géneros,
  plataformas, productores). Con valores casi únicos (títulos) pierde: añade el
  diccionario sin ahorrar nada.

  Regla de orden: mide primero con memory_usage(deep=True), optimiza después.
  Optimizar sin medir es adivinar.""")


▸ Memoria antes y después de category
{'antes': '1.35 MB', 'después': '0.45 MB', 'ahorro': '67%'}

▸ downcast de enteros
{'int64': 8132, 'int16': 2132}

  category gana cuando hay MUCHAS filas y POCOS valores distintos (géneros,
  plataformas, productores). Con valores casi únicos (títulos) pierde: añade el
  diccionario sin ahorrar nada.

  Regla de orden: mide primero con memory_usage(deep=True), optimiza después.
  Optimizar sin medir es adivinar.


---

<a id="parte-f"></a>
# Parte F · MongoDB

<a id="sec-24"></a>
## 24. Conceptos: por qué documental y no relacional
<sub>[↑ índice](#indice)</sub>

| Relacional | MongoDB | Qué cambia |
|---|---|---|
| tabla | **colección** | sin esquema fijo declarado |
| fila | **documento** (BSON) | puede anidar objetos y arrays |
| columna | **campo** | distintos documentos, distintos campos |
| `JOIN` | `$lookup` o **embebido** | el join es caro: a veces conviene duplicar |
| `CREATE INDEX` | `create_index` | igual de obligatorio |

> **Idea clave — la regla de modelado documental.** *Lo que se lee junto, se guarda
> junto.* Si cada vez que muestras un stand necesitas también sus etiquetas, embebe las
> etiquetas en el documento del stand. Si las etiquetas se consultan por sí solas o
> cambian mucho, sepáralas y referencia.
>
> El límite duro: un documento BSON no puede pasar de **16 MB**, y un array que crece
> sin tope (todos los asistentes de la feria dentro del documento del stand) es un
> antipatrón — Kleppmann lo trata como el caso clásico de modelo que no escala.

### Esquema flexible no significa "sin esquema"

Que Mongo **acepte** cualquier documento no significa que tu aplicación lo tolere. El
esquema sigue existiendo: lo que cambia es **dónde se valida**.

```
RELACIONAL:  el esquema lo valida la BASE  (CREATE TABLE lo impone)
DOCUMENTAL:  el esquema lo valida TU CÓDIGO (o $jsonSchema, si lo declaras)
```

Si nadie lo valida, el esquema lo descubre el que lea los datos en tres meses. Por eso
la Parte E (asserts) **precede** a esta parte y no es opcional.

In [40]:
# --- Conexión tolerante: Atlas -> local -> sin Mongo ---
# El notebook debe poder ejecutarse completo aunque no haya servidor: así Cris y
# Sofi pueden leerlo en su máquina sin instalar nada.
HAY_MONGO = False
cli = db = None
try:
    from pymongo import MongoClient, ASCENDING, DESCENDING, TEXT
    from pymongo.errors import ServerSelectionTimeoutError

    # serverSelectionTimeoutMS bajo: fallar en 1.5 s en vez de colgarse 30
    cli = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=1500)
    cli.admin.command("ping")                     # la única forma de confirmar conexión real
    db = cli["tutorial_feria"]
    HAY_MONGO = True
    print("✓ MongoDB conectado en localhost:27017")
except Exception as e:
    print(f"MongoDB no disponible ({type(e).__name__}). Probando simulador…")
    try:
        import mongomock
        cli = mongomock.MongoClient()             # Mongo EN MEMORIA, sin servidor
        db = cli["tutorial_feria"]
        HAY_MONGO = True
        print("✓ Usando mongomock (en memoria): el código es idéntico")
    except ImportError:
        print("Sin mongomock: pip install mongomock")
        print("Las celdas de Mongo se saltarán solas con 'if HAY_MONGO'")

ver("¿Hay Mongo?", HAY_MONGO)
print("""
  ⚠️ MongoClient() NO conecta: la conexión es diferida (lazy). Si no haces
  ping, el error aparece recién en la primera operación real, en otra celda, y
  el traceback no menciona la conexión. El ping es el que te da el error donde
  corresponde.

  ⚠️ mongomock NO implementa todo: le faltan $mergeObjects, $setIntersection,
  $round y el índice de texto $text. Lo que corre en el simulador puede fallar
  distinto en Atlas, y al revés. Para la entrega, prueba en un Mongo real.""")

MongoDB no disponible (ServerSelectionTimeoutError). Probando simulador…
Sin mongomock: pip install mongomock
Las celdas de Mongo se saltarán solas con 'if HAY_MONGO'

▸ ¿Hay Mongo?
False

  ⚠️ MongoClient() NO conecta: la conexión es diferida (lazy). Si no haces
  ping, el error aparece recién en la primera operación real, en otra celda, y
  el traceback no menciona la conexión. El ping es el que te da el error donde
  corresponde.

  ⚠️ mongomock NO implementa todo: le faltan $mergeObjects, $setIntersection,
  $round y el índice de texto $text. Lo que corre en el simulador puede fallar
  distinto en Atlas, y al revés. Para la entrega, prueba en un Mongo real.


---

<a id="sec-25"></a>
## 25. De DataFrame a documentos
<sub>[↑ índice](#indice)</sub>

> ⚠️ **El error que te va a pasar.** BSON **no sabe** qué es `np.int64`, `NaN` ni `NaT`.
> `NaN` se inserta como el float IEEE `nan` — queda en la base, no lanza error, y
> aparece en los resultados como un valor que no es ni nulo ni número. `NaT` lanza
> `NaTType does not support utcoffset`. La conversión hay que hacerla **a propósito**.

In [41]:
def limpiar_para_bson(v):
    """Convierte UN valor de pandas/NumPy a algo que BSON entienda.

    El orden de los if importa: pd.isna() debe ir ANTES de las conversiones de
    tipo, porque NaT es un datetime a efectos de isinstance y np.nan es float.
    """
    if v is None:
        return None
    # pd.isna cubre NaN, NaT y pd.NA de una sola vez
    if isinstance(v, (float, np.floating)) and (pd.isna(v)):
        return None
    if v is pd.NaT or (not isinstance(v, (list, dict, np.ndarray)) and pd.isna(v)):
        return None
    if isinstance(v, (np.integer,)):      # np.int64 -> int de Python
        return int(v)
    if isinstance(v, (np.floating,)):     # np.float64 -> float de Python
        return float(v)
    if isinstance(v, (np.bool_,)):        # np.bool_ -> bool de Python
        return bool(v)
    if isinstance(v, np.ndarray):         # array -> lista
        return v.tolist()
    if isinstance(v, pd.Timestamp):       # Timestamp -> datetime nativo
        return v.to_pydatetime()
    if isinstance(v, (list, tuple)):      # recursión en listas
        return [limpiar_para_bson(x) for x in v]
    if isinstance(v, dict):               # recursión en dicts
        return {k: limpiar_para_bson(x) for k, x in v.items()}
    return v


def a_documentos(d: pd.DataFrame) -> list:
    """DataFrame -> lista de dicts listos para insert_many."""
    return [{k: limpiar_para_bson(v) for k, v in fila.items()}
            for fila in d.to_dict("records")]      # "records" = una fila por dict


sucio = pd.DataFrame({
    "id": pd.array([1, 2], dtype="Int64"),
    "precio": [np.float64(9.99), np.nan],                  # NaN explícito
    "fecha": pd.to_datetime(["2026-11-01", None]),         # NaT en la segunda
    "tags": [["a", "b"], []],
    "n": [np.int64(5), np.int64(7)],                       # np.int64 que BSON rechaza
})
ver("DataFrame con todos los tipos problemáticos", sucio)
ver("Tipos originales", sucio.dtypes.to_dict())

docs_ok = a_documentos(sucio)
ver("Documentos convertidos", docs_ok)
ver("Tipos de Python resultantes",
    {k: type(v).__name__ for k, v in docs_ok[0].items()})
ver("Los nulos quedaron como None, no como NaN/NaT",
    {k: v for k, v in docs_ok[1].items() if v is None})

# La prueba definitiva: si json.dumps lo serializa, BSON lo acepta
import json
ver("¿Serializable?", json.dumps(docs_ok, default=str)[:120] + "…")
print("""
  ⚠️ El caso NaT es el peor porque el mensaje de error no ayuda:
  "NaTType does not support utcoffset". No menciona Mongo, ni la columna, ni la
  fila. Si lo ves, el culpable es una columna datetime con nulos.

  ⚠️ Y hay un detalle extra con las fechas: si en vez de limpiar valor a valor
  intentas arreglar la COLUMNA con .map(), pandas vuelve a inferir el dtype
  datetime64 y los None se convierten de nuevo en NaT. Para forzarlo hay que
  construir la Series con dtype=object explícitamente.""")


▸ DataFrame con todos los tipos problemáticos


,id,precio,fecha,tags,n
0,1,9.99,2026-11-01,"[a, b]",5
1,2,NaN,NaT,[],7



▸ Tipos originales
{'id': Int64Dtype(), 'precio': dtype('float64'), 'fecha': dtype('<M8[us]'), 'tags': dtype('O'), 'n': dtype('int64')}

▸ Documentos convertidos
[{'id': 1, 'precio': 9.99, 'fecha': datetime.datetime(2026, 11, 1, 0, 0), 'tags': ['a', 'b'], 'n': 5}, {'id': 2, 'precio': None, 'fecha': None, 'tags': [], 'n': 7}]

▸ Tipos de Python resultantes
{'id': 'int', 'precio': 'float', 'fecha': 'datetime', 'tags': 'list', 'n': 'int'}

▸ Los nulos quedaron como None, no como NaN/NaT
{'precio': None, 'fecha': None}

▸ ¿Serializable?
[{"id": 1, "precio": 9.99, "fecha": "2026-11-01 00:00:00", "tags": ["a", "b"], "n": 5}, {"id": 2, "precio": null, "fecha…

  ⚠️ El caso NaT es el peor porque el mensaje de error no ayuda:
  "NaTType does not support utcoffset". No menciona Mongo, ni la columna, ni la
  fila. Si lo ves, el culpable es una columna datetime con nulos.

  ⚠️ Y hay un detalle extra con las fechas: si en vez de limpiar valor a valor
  intentas arreglar la COLUMNA con .map(), pan

---

<a id="sec-26"></a>
## 26. CRUD
<sub>[↑ índice](#indice)</sub>

> ⚠️ **`update_one` sin `$set`** reemplaza el documento completo y borra los campos que
> no pusiste. `{"precio": 10}` como actualización deja un documento con **solo** `precio`.

In [42]:
if HAY_MONGO:
    col = db["obras"]
    col.delete_many({})                 # limpiar primero: hace la celda IDEMPOTENTE

    base = pd.DataFrame({
        "obra_id": [1, 2, 3, 4],
        "titulo": ["Alfa", "Beta", "Gamma", "Delta"],
        "stand_id": ["s1", "s1", "s2", "s2"],
        "tags": [["accion", "rol"], ["accion"], ["terror"], ["rol", "terror"]],
        "precio": [10.0, 20.0, 30.0, 40.0],
    })

    r = col.insert_many(a_documentos(base))
    ver("insert_many: ids generados", len(r.inserted_ids))
    ver("count_documents", col.count_documents({}))
    ver("find_one", col.find_one({"obra_id": 1}, {"_id": 0}))   # {"_id":0} oculta el _id

    # UPDATE con $set: modifica SOLO el campo indicado
    col.update_one({"obra_id": 1}, {"$set": {"precio": 12.5}})
    ver("Tras update_one con $set", col.find_one({"obra_id": 1}, {"_id": 0}))

    col.update_many({"stand_id": "s1"}, {"$set": {"destacado": True}})
    ver("update_many: documentos afectados", col.count_documents({"destacado": True}))

    # $addToSet añade sin duplicar; $push añadiría aunque ya esté
    col.update_one({"obra_id": 2}, {"$addToSet": {"tags": "indie"}})
    col.update_one({"obra_id": 2}, {"$addToSet": {"tags": "accion"}})   # ya existe: no hace nada
    ver("$addToSet no duplica", col.find_one({"obra_id": 2}, {"_id": 0, "tags": 1}))

    col.update_one({"obra_id": 3}, {"$inc": {"precio": 5}})      # incremento atómico
    ver("$inc", col.find_one({"obra_id": 3}, {"_id": 0, "precio": 1}))

    # UPSERT: actualiza si existe, inserta si no. La base de un ETL idempotente.
    col.update_one({"obra_id": 99}, {"$set": {"titulo": "Nueva"}}, upsert=True)
    ver("Tras el upsert", col.count_documents({}))
    col.update_one({"obra_id": 99}, {"$set": {"titulo": "Nueva v2"}}, upsert=True)
    ver("El upsert repetido NO duplica", col.count_documents({}))

    col.delete_one({"obra_id": 99})
    ver("Tras delete_one", col.count_documents({}))
else:
    print("Sin Mongo: celda saltada")

Sin Mongo: celda saltada


In [43]:
if HAY_MONGO:
    from pymongo import ReplaceOne

    # --- CARGA IDEMPOTENTE: la operación central del ETL ---
    # replace_one con upsert=True sobre una clave de negocio: correr el ETL dos
    # veces deja la base igual que correrlo una. Es lo que permite reintentar
    # un ETL fallido sin duplicar nada.
    ops = [ReplaceOne({"obra_id": d["obra_id"]}, d, upsert=True)
           for d in a_documentos(base)]

    try:
        res = col.bulk_write(ops, ordered=False)       # ordered=False: no se detiene al primer fallo
        ver("bulk_write", {"upserted": res.upserted_count, "modified": res.modified_count})
    except TypeError as e:
        # ⚠️ Incompatibilidad real pymongo 4.19 / mongomock 4.3:
        #    "add_replace() got an unexpected keyword argument 'sort'".
        #    No es un bug tuyo: es una firma que cambió. El fallback hace lo
        #    mismo con N llamadas en vez de una.
        print(f"bulk_write no soportado aquí ({e}); usando replace_one uno por uno")
        for d in a_documentos(base):
            col.replace_one({"obra_id": d["obra_id"]}, d, upsert=True)
        ver("Fallback completado", col.count_documents({}))

    ver("Total tras recargar lo mismo (no debe crecer)", col.count_documents({}))

    # create_index con unique=True: la base IMPONE la unicidad
    col.create_index([("obra_id", 1)], unique=True, name="uk_obra")
    try:
        col.insert_one({"obra_id": 1, "titulo": "duplicado"})
    except Exception as e:
        ver("El índice único rechaza el duplicado", type(e).__name__)
else:
    print("Sin Mongo: celda saltada")
print("""
  Idempotencia en una frase: insert_many duplica si lo corres dos veces;
  replace_one(clave, doc, upsert=True) no. Esa es la diferencia entre un ETL
  que puedes reintentar y uno que te obliga a borrar la base cada vez.""")

Sin Mongo: celda saltada

  Idempotencia en una frase: insert_many duplica si lo corres dos veces;
  replace_one(clave, doc, upsert=True) no. Esa es la diferencia entre un ETL
  que puedes reintentar y uno que te obliga a borrar la base cada vez.


---

<a id="sec-27"></a>
## 27. Consultas y agregación
<sub>[↑ índice](#indice)</sub>

El *aggregation pipeline* es una **secuencia de etapas**, cada una transformando la
salida de la anterior:

```
$match   → filtra documentos         (lo primero, SIEMPRE: reduce el volumen)
$unwind  → un documento por elemento del array   (≡ explode de pandas)
$group   → agrupa y acumula          (≡ groupby)
$sort    → ordena
$limit   → corta
$lookup  → join con otra colección
$project → elige/renombra campos
```

> **Por qué `$match` va primero.** Cada etapa procesa lo que le llega. Un `$match` al
> final filtra después de haber desenrollado y agrupado **todo**; al principio, descarta
> antes de gastar. Además, solo un `$match` inicial puede usar un **índice**.

In [44]:
if HAY_MONGO:
    ver("Igualdad simple", list(col.find({"stand_id": "s1"}, {"_id": 0, "titulo": 1})))
    ver("$gt / $lte: comparadores",
        list(col.find({"precio": {"$gt": 15, "$lte": 35}}, {"_id": 0, "titulo": 1, "precio": 1})))
    ver("$in: pertenencia", list(col.find({"stand_id": {"$in": ["s1", "s2"]}}, {"_id": 0, "titulo": 1})))
    # Sobre un ARRAY, la igualdad simple busca si el array CONTIENE el valor
    ver("tags: 'accion' (el array contiene)", list(col.find({"tags": "accion"}, {"_id": 0, "titulo": 1})))
    ver("$all: contiene TODOS", list(col.find({"tags": {"$all": ["rol", "terror"]}}, {"_id": 0, "titulo": 1})))
    ver("$size: longitud exacta del array", list(col.find({"tags": {"$size": 2}}, {"_id": 0, "titulo": 1})))
    ver("$exists: el campo está presente", col.count_documents({"destacado": {"$exists": True}}))
    ver("$regex: texto (sin índice de texto)",
        list(col.find({"titulo": {"$regex": "^A", "$options": "i"}}, {"_id": 0, "titulo": 1})))
    ver("sort + limit", list(col.find({}, {"_id": 0, "titulo": 1, "precio": 1})
                             .sort("precio", -1).limit(2)))
    ver("distinct: valores únicos de un campo", sorted(col.distinct("stand_id")))
else:
    print("Sin Mongo: celda saltada")
print("""
  ⚠️ {"tags": "accion"} NO significa "tags es igual a 'accion'": sobre un array,
  Mongo interpreta la igualdad como "contiene". Para exigir el array exacto
  tendrías que escribir {"tags": ["accion", "rol"]}, que es una consulta
  distinta y sensible al ORDEN.""")

Sin Mongo: celda saltada

  ⚠️ {"tags": "accion"} NO significa "tags es igual a 'accion'": sobre un array,
  Mongo interpreta la igualdad como "contiene". Para exigir el array exacto
  tendrías que escribir {"tags": ["accion", "rol"]}, que es una consulta
  distinta y sensible al ORDEN.


In [45]:
if HAY_MONGO:
    # Pipeline 1: etiquetas más frecuentes  (≡ explode + value_counts)
    p1 = [
        {"$unwind": "$tags"},                                     # una fila por tag
        {"$group": {"_id": "$tags", "n": {"$sum": 1}}},           # contar por tag
        {"$sort": {"n": -1}},                                     # de mayor a menor
        {"$limit": 5},
    ]
    ver("Top etiquetas", list(col.aggregate(p1)))

    # Pipeline 2: resumen por stand  (≡ groupby().agg())
    p2 = [
        {"$group": {"_id": "$stand_id",
                    "obras": {"$sum": 1},
                    "precio_prom": {"$avg": "$precio"},
                    "precio_max": {"$max": "$precio"},
                    "titulos": {"$push": "$titulo"}}},            # $push acumula en array
        {"$sort": {"obras": -1}},
    ]
    ver("Resumen por stand", list(col.aggregate(p2)))

    # Pipeline 3: $lookup — el join documental
    db["stands"].delete_many({})
    db["stands"].insert_many([{"stand_id": "s1", "nombre": "Acme", "pasillo": "A"},
                              {"stand_id": "s2", "nombre": "Zeta", "pasillo": "B"}])
    p3 = [
        {"$lookup": {"from": "stands",             # colección a unir
                     "localField": "stand_id",     # campo de ESTA colección
                     "foreignField": "stand_id",   # campo de la otra
                     "as": "stand"}},              # el resultado llega como ARRAY
        # $unwind convierte ese array de 1 elemento en un objeto plano
        {"$unwind": "$stand"},
        {"$project": {"_id": 0, "titulo": 1,
                      "stand": "$stand.nombre",    # notación de punto para anidados
                      "pasillo": "$stand.pasillo"}},
    ]
    ver("Join con $lookup", list(col.aggregate(p3)))

    # Pipeline 4: similitud calculada EN LA BASE
    ref = {"accion", "rol"}
    p4 = [
        {"$match": {"obra_id": {"$ne": 1}}},                     # excluir la referencia
        {"$project": {
            "_id": 0, "titulo": 1, "tags": 1,
            # $filter + $in + $size: intersección portable, sin $setIntersection
            "comunes": {"$size": {"$filter": {"input": "$tags", "as": "t",
                                              "cond": {"$in": ["$$t", list(ref)]}}}},
            "n_tags": {"$size": "$tags"},
        }},
        {"$match": {"comunes": {"$gt": 0}}},
        {"$sort": {"comunes": -1}},
    ]
    ver("Similitud dentro de Mongo", list(col.aggregate(p4)))
else:
    print("Sin Mongo: celda saltada")
print("""
  ⚠️ $setIntersection, $mergeObjects y $round NO existen en mongomock. El
  pipeline 4 usa $filter + $in + $size, que es más verboso pero corre igual en
  el simulador y en Atlas. Si solo pruebas en Atlas, tu notebook no le va a
  funcionar a quien no tenga servidor.

  ⚠️ $lookup devuelve un ARRAY (puede haber varias coincidencias), de ahí el
  $unwind. Y es la etapa MÁS CARA del pipeline: si un join se repite en toda
  consulta, esa es la señal de que los datos debían ir embebidos.""")

Sin Mongo: celda saltada

  ⚠️ $setIntersection, $mergeObjects y $round NO existen en mongomock. El
  pipeline 4 usa $filter + $in + $size, que es más verboso pero corre igual en
  el simulador y en Atlas. Si solo pruebas en Atlas, tu notebook no le va a
  funcionar a quien no tenga servidor.

  ⚠️ $lookup devuelve un ARRAY (puede haber varias coincidencias), de ahí el
  $unwind. Y es la etapa MÁS CARA del pipeline: si un join se repite en toda
  consulta, esa es la señal de que los datos debían ir embebidos.


---

<a id="sec-28"></a>
## 28. Índices y `explain()`
<sub>[↑ índice](#indice)</sub>

> **Idea clave.** Un índice es una **estructura ordenada** (B-tree) que evita recorrer
> toda la colección. Sin índice, Mongo hace `COLLSCAN`: lee **todos** los documentos.
> Con índice, `IXSCAN`: baja por el árbol en tiempo logarítmico.
>
> El costo: cada índice ocupa espacio y **encarece las escrituras**, porque hay que
> mantenerlo actualizado. Por eso no se indexa "todo por si acaso".

In [46]:
if HAY_MONGO:
    muchos = pd.DataFrame({
        "obra_id": np.arange(5000),
        "stand_id": [f"s{i % 50}" for i in range(5000)],
        "precio": np.random.default_rng(7).uniform(5, 100, 5000).round(2),
        "tags": [["accion"] if i % 2 else ["rol"] for i in range(5000)],
    })
    big = db["obras_grande"]
    big.delete_many({})
    big.insert_many(a_documentos(muchos))
    ver("Documentos insertados", big.count_documents({}))

    def plan(c, filtro):
        """Extrae del explain() solo lo que importa: etapa y documentos examinados."""
        try:
            e = c.find(filtro).explain()
        except Exception as err:
            # mongomock implementa explain() de forma parcial o no lo implementa:
            # devolvemos el motivo en vez de romper el notebook.
            return {"explain no disponible": type(err).__name__}
        ganador = e.get("queryPlanner", {}).get("winningPlan", {})
        ejec = e.get("executionStats", {})
        # El plan puede venir anidado dentro de inputStage: hay que bajar
        etapa = ganador.get("stage")
        nodo = ganador
        while "inputStage" in nodo:
            nodo = nodo["inputStage"]
            etapa = f"{etapa} -> {nodo.get('stage')}"
        return {"etapa": etapa,
                "examinados": ejec.get("totalDocsExamined"),
                "devueltos": ejec.get("nReturned")}

    ver("SIN índice", plan(big, {"stand_id": "s7"}))
    big.create_index([("stand_id", 1)], name="ix_stand")     # 1 = ascendente
    ver("CON índice", plan(big, {"stand_id": "s7"}))
    ver("Índices de la colección", [i["name"] for i in big.list_indexes()])

    # ÍNDICE COMPUESTO: el ORDEN de los campos decide qué consultas sirve
    big.create_index([("stand_id", 1), ("precio", -1)], name="ix_stand_precio")
    ver("Compuesto: filtra por stand y ordena por precio",
        plan(big, {"stand_id": "s7", "precio": {"$gt": 50}}))

    # ÍNDICE MULTIKEY: sobre un array, indexa cada elemento
    big.create_index([("tags", 1)], name="ix_tags")
    ver("Multikey sobre el array de tags", plan(big, {"tags": "accion"}))
else:
    print("Sin Mongo: celda saltada")
print("""
  Lo que hay que leer en explain():
    COLLSCAN            -> recorrió TODO. Falta un índice.
    IXSCAN              -> usó el índice. Bien.
    totalDocsExamined   -> si es ≈ a la colección completa, el índice no sirvió
    nReturned           -> lo ideal es examinados ≈ devueltos

  ⚠️ Si arriba dice "explain no disponible: AttributeError", es porque estás en
  mongomock: el simulador NO implementa explain(). Eso significa que esta celda
  NO es evidencia del efecto del índice — es solo el código correcto para
  pedirla. La celda siguiente mide el mecanismo de otra forma, y al final de la
  sección están los comandos exactos para obtener la evidencia real.""")

Sin Mongo: celda saltada

  Lo que hay que leer en explain():
    COLLSCAN            -> recorrió TODO. Falta un índice.
    IXSCAN              -> usó el índice. Bien.
    totalDocsExamined   -> si es ≈ a la colección completa, el índice no sirvió
    nReturned           -> lo ideal es examinados ≈ devueltos

  ⚠️ Si arriba dice "explain no disponible: AttributeError", es porque estás en
  mongomock: el simulador NO implementa explain(). Eso significa que esta celda
  NO es evidencia del efecto del índice — es solo el código correcto para
  pedirla. La celda siguiente mide el mecanismo de otra forma, y al final de la
  sección están los comandos exactos para obtener la evidencia real.


### El mecanismo, medido sin servidor

Un `explain()` que no corre no enseña nada. Pero el **mecanismo** del índice —bajar por
una estructura ordenada en vez de recorrer todo— sí se puede medir aquí, contando
comparaciones:

```
COLLSCAN      recorre n elementos          →  O(n)      comparaciones ≈ n
IXSCAN        baja por un árbol ordenado   →  O(log n)  comparaciones ≈ log₂ n
```

In [47]:
import bisect

def comparaciones_scan(datos: list, objetivo) -> int:
    """Búsqueda lineal: cuenta cuántos elementos tuvo que mirar (≡ COLLSCAN)."""
    c = 0
    for x in datos:
        c += 1
        if x == objetivo:
            return c
    return c

def comparaciones_binaria(ordenados: list, objetivo) -> int:
    """Búsqueda binaria sobre datos ORDENADOS: cuenta los pasos (≡ IXSCAN).

    Un B-tree hace exactamente esto, pero con nodos de muchas claves en vez de
    dos mitades — de ahí que la base del logaritmo sea grande y el árbol, bajo.
    """
    lo, hi, c = 0, len(ordenados), 0
    while lo < hi:
        c += 1
        mid = (lo + hi) // 2
        if ordenados[mid] == objetivo:
            return c
        if ordenados[mid] < objetivo:
            lo = mid + 1
        else:
            hi = mid
    return c

filas = []
for n in (1_000, 10_000, 100_000, 1_000_000):
    datos = list(range(n))
    objetivo = n - 1                       # el PEOR caso para el scan: el último
    filas.append({
        "n": n,
        "scan (COLLSCAN)": comparaciones_scan(datos, objetivo),
        "binaria (IXSCAN)": comparaciones_binaria(datos, objetivo),
        "log2(n) teórico": round(np.log2(n), 1),
    })
tabla = pd.DataFrame(filas)
tabla["factor"] = (tabla["scan (COLLSCAN)"] / tabla["binaria (IXSCAN)"]).round(0).astype(int)
ver("Comparaciones necesarias para encontrar un elemento", tabla)
print(f"""
  Lee la última columna: al pasar de 1.000 a 1.000.000 de documentos el scan
  necesita 1.000 veces más trabajo, y el índice solo {tabla.loc[3,'binaria (IXSCAN)']/tabla.loc[0,'binaria (IXSCAN)']:.1f}× más. Esa es la razón
  por la que un índice no "acelera un poco": cambia la CLASE de complejidad.

  Y la columna 'log2(n) teórico' calza con las comparaciones medidas — eso
  confirma que la búsqueda binaria es O(log n), no una impresión.

  ⚠️ Lo que este experimento NO demuestra: que MongoDB use el índice en TU
  consulta. Eso depende del plan que elija el optimizador, y solo lo dice
  explain(). El mecanismo es universal; la aplicación concreta hay que medirla.""")


▸ Comparaciones necesarias para encontrar un elemento


,n,scan (COLLSCAN),binaria (IXSCAN),log2(n) teórico,factor
0,1000,1000,9,10.0,111
1,10000,10000,13,13.3,769
2,100000,100000,16,16.6,6250
3,1000000,1000000,19,19.9,52632



  Lee la última columna: al pasar de 1.000 a 1.000.000 de documentos el scan
  necesita 1.000 veces más trabajo, y el índice solo 2.1× más. Esa es la razón
  por la que un índice no "acelera un poco": cambia la CLASE de complejidad.

  Y la columna 'log2(n) teórico' calza con las comparaciones medidas — eso
  confirma que la búsqueda binaria es O(log n), no una impresión.

  ⚠️ Lo que este experimento NO demuestra: que MongoDB use el índice en TU
  consulta. Eso depende del plan que elija el optimizador, y solo lo dice
  explain(). El mecanismo es universal; la aplicación concreta hay que medirla.


### 🔧 La evidencia real: cómo obtenerla

En un Mongo de verdad (Atlas o un contenedor local), corre esto y **guarda la salida**
—es lo que vas a mostrar en la entrega:

```bash
docker run -d -p 27017:27017 --name mongo-feria mongo:7
```

```python
# Volver a correr el notebook: HAY_MONGO será True contra el servidor real
plan(big, {"stand_id": "s7"})              # ANTES del índice  -> COLLSCAN
big.create_index([("stand_id", 1)])
plan(big, {"stand_id": "s7"})              # DESPUÉS           -> IXSCAN
```

Lo que debes poder reportar, con números: `totalDocsExamined` antes, después, y el
cociente. Si no baja, el índice está mal elegido y hay que revisar la regla ESR.

> ⚠️ **Regla ESR — el orden en un índice compuesto importa.**
> **E**quality → **S**ort → **R**ange. El índice `(stand_id, precio)` sirve para
> filtrar por stand y ordenar por precio; `(precio, stand_id)` **no** sirve para lo
> mismo, aunque tenga los mismos campos.

---

<a id="sec-29"></a>
## 29. Réplicas y particionamiento (visión de arquitectura)
<sub>[↑ índice](#indice)</sub>

```
REPLICACIÓN (disponibilidad)        PARTICIONAMIENTO / SHARDING (escala)
   ┌─────────┐                         ┌──────────┐  ┌──────────┐
   │ PRIMARY │◄── escrituras           │ shard A  │  │ shard B  │
   └────┬────┘                         │ tags a-m │  │ tags n-z │
   ┌────┴────┬────────┐                └──────────┘  └──────────┘
   ▼         ▼        ▼                 ↑ la shard key decide el reparto
┌──────┐ ┌──────┐ ┌──────┐
│SEC. 1│ │SEC. 2│ │SEC. 3│  ← lecturas
└──────┘ └──────┘ └──────┘
Misma copia 3 veces              Datos DISTINTOS en cada nodo
```

| | Replicación | Sharding |
|---|---|---|
| Resuelve | caída de un nodo, lecturas | volumen y escrituras |
| Los datos | **duplicados** | **repartidos** |
| Costo | ×3 almacenamiento | complejidad y joins entre shards |

> **El trade-off que el profesor va a preguntar.** `w=1` confirma la escritura cuando la
> recibe el primario: rápido, pero si el primario cae antes de replicar, el dato **se
> pierde**. `w="majority"` espera a la mayoría: más lento, y la escritura sobrevive a la
> caída de un nodo. No hay opción "correcta": hay una decisión que depende de si el dato
> es un *like* o un pago.
>
> Una **shard key mal elegida** produce *hot spots*: si repartes por fecha y todos
> escriben hoy, todas las escrituras caen en el mismo shard y el clúster entero va a la
> velocidad de un nodo. El término técnico para ese síntoma es **subutilización del
> clúster**.

In [48]:
if HAY_MONGO:
    try:
        # La "topología" dice si hay réplicas de verdad o es un nodo solo
        info = cli.admin.command("hello")
        ver("Topología", {k: info.get(k) for k in ("isWritablePrimary", "setName", "hosts")})
    except Exception as e:
        ver("hello no disponible (normal en mongomock/standalone)", type(e).__name__)

    try:
        from pymongo import WriteConcern, ReadPreference
        # w="majority": confirmar solo cuando la MAYORÍA del replica set lo tenga
        segura = db.get_collection("obras", write_concern=WriteConcern(w="majority"))
        segura.update_one({"obra_id": 1}, {"$set": {"revisado": True}})
        ver("Escritura con w='majority'", "aceptada")
    except Exception as e:
        ver("WriteConcern no aplicable en este entorno", type(e).__name__)
else:
    print("Sin Mongo: celda saltada")
print("""
  🎯 Para tu proyecto (Entrega 3 y la defensa)
     Prepara la respuesta a: "¿por qué MongoDB y no PostgreSQL?"
     La respuesta débil: "porque es NoSQL y es más rápido".
     La respuesta defendible nombra el TRADE-OFF concreto:
       · esquema flexible -> las 3 fuentes tienen campos distintos y no
         queremos un ALTER TABLE por fuente
       · documento embebido -> stand + sus etiquetas se leen juntos, y el
         JOIN desaparece de la consulta más frecuente
       · lo que SE PIERDE -> transacciones multi-documento cómodas, integridad
         referencial impuesta por la base, y SQL para consultas ad hoc
     Admitir lo que se pierde es lo que hace creíble lo que se gana.""")

Sin Mongo: celda saltada

  🎯 Para tu proyecto (Entrega 3 y la defensa)
     Prepara la respuesta a: "¿por qué MongoDB y no PostgreSQL?"
     La respuesta débil: "porque es NoSQL y es más rápido".
     La respuesta defendible nombra el TRADE-OFF concreto:
       · esquema flexible -> las 3 fuentes tienen campos distintos y no
         queremos un ALTER TABLE por fuente
       · documento embebido -> stand + sus etiquetas se leen juntos, y el
         JOIN desaparece de la consulta más frecuente
       · lo que SE PIERDE -> transacciones multi-documento cómodas, integridad
         referencial impuesta por la base, y SQL para consultas ad hoc
     Admitir lo que se pierde es lo que hace creíble lo que se gana.


---

<a id="parte-g"></a>
# Parte G · Cierre

<a id="sec-30"></a>
## 30. Plantilla de ETL completo
<sub>[↑ índice](#indice)</sub>

```
     EXTRAER        TRANSFORMAR        VALIDAR          CARGAR
   ┌─────────┐    ┌───────────┐    ┌──────────┐    ┌───────────┐
   │  CSV    │───▶│ normaliza │───▶│ asserts  │───▶│  Mongo    │
   │  JSON   │    │ mapea     │    │ informe  │    │ (upsert)  │
   └─────────┘    │ agrupa    │    └──────────┘    └───────────┘
     ▲            └───────────┘          │
     │                                   ▼
  perfilar()                   leídos = cargados + rechazados
```

**Una función por etapa**, y cada una devuelve un informe. No es burocracia: es lo que
permite que cuando el ETL falle sepas **en qué etapa** y **con qué dato**.

In [49]:
def extraer(ruta_o_texto, **kw) -> pd.DataFrame:
    """ETAPA 1 — leer sin transformar nada.

    Mantener la extracción "tonta" permite volver al dato original cuando una
    transformación resulta sospechosa.
    """
    from io import StringIO
    fuente = StringIO(ruta_o_texto) if isinstance(ruta_o_texto, str) and "\n" in ruta_o_texto \
             else ruta_o_texto
    return pd.read_csv(fuente, **kw)


def transformar(d: pd.DataFrame, mapa: dict, catalogo: set) -> tuple:
    """ETAPA 2 — normalizar, mapear y agrupar. Devuelve (datos, informe).

    El informe es lo que convierte un filtro silencioso en una cifra auditable.
    """
    import unicodedata
    inf = Counter(leidos=len(d))

    def sin_tildes(x):
        if not isinstance(x, str):
            return x
        return "".join(c for c in unicodedata.normalize("NFKD", x)
                       if not unicodedata.combining(c))

    d = d.copy()

    # Normalizar el productor: resuelve " Acme " vs "acme" (§9)
    d["productor"] = (d["productor"].map(sin_tildes).str.strip().str.lower()
                                    .str.replace(r"\s+", " ", regex=True))

    # Regla de negocio 1: sin productor no hay stand -> se RECHAZA y se cuenta
    antes = len(d)
    d = d[d["productor"].notna() & (d["productor"] != "")]
    inf["rechazo_sin_productor"] = antes - len(d)

    # Regla de negocio 2: un título por productor (duplicados de carga)
    antes = len(d)
    d = d.drop_duplicates(subset=["titulo", "productor"])
    inf["rechazo_duplicado"] = antes - len(d)

    # Etiquetas: texto -> lista -> normalizadas -> mapeadas al catálogo oficial
    d["tags"] = (d["generos"].fillna("").str.split(",")
                 .map(lambda xs: [x.strip().lower().replace(" ", "-") for x in xs if x.strip()]))
    d["tags"] = d["tags"].map(lambda xs: [mapa.get(x, x) for x in xs])

    # Lo que no está en el catálogo se descarta, PERO se cuenta
    desconocidas = {t for xs in d["tags"] for t in xs} - catalogo
    inf["etiquetas_desconocidas"] = len(desconocidas)
    inf["ejemplos_desconocidas"] = sorted(desconocidas)[:5]
    d["tags"] = d["tags"].map(lambda xs: sorted({t for t in xs if t in catalogo}))

    # Regla de negocio 3: sin etiquetas válidas no se puede recomendar
    antes = len(d)
    d = d[d["tags"].map(len) > 0]
    inf["rechazo_sin_etiquetas"] = antes - len(d)

    return d.reset_index(drop=True), inf


def validar_etl(d: pd.DataFrame, catalogo: set) -> None:
    """ETAPA 3 — afirmar las invariantes. Falla RUIDOSAMENTE si no se cumplen."""
    assert not d.empty, "no quedaron filas tras transformar"
    assert d["titulo"].notna().all(), "títulos nulos"
    assert d["tags"].map(len).gt(0).all(), "filas sin etiquetas"
    assert all(t in catalogo for xs in d["tags"] for t in xs), "etiqueta fuera del catálogo"
    assert not d.duplicated(subset=["titulo", "productor"]).any(), "duplicados sobrevivientes"


def cargar(d: pd.DataFrame, coleccion, clave: str = "titulo") -> int:
    """ETAPA 4 — carga IDEMPOTENTE: replace_one + upsert sobre la clave de negocio."""
    if coleccion is None:
        return 0
    n = 0
    for doc in a_documentos(d):
        coleccion.replace_one({clave: doc[clave]}, doc, upsert=True)
        n += 1
    return n

In [50]:
# --- El ETL corriendo de punta a punta ---
csv_demo = """titulo;productor;generos
Alfa;  Acme ;Action, Fantasy
Beta;acme;sci-fi
Gamma;Zeta Co;Role-Playing, Horror
Gamma;Zeta Co;Role-Playing, Horror
Delta;;Action
Epsilon;Omega;cosa-inventada
Zeta;Omega;Action, cosa-inventada
"""

MAPA = {"action": "accion", "fantasy": "fantasia", "sci-fi": "ciencia-ficcion",
        "role-playing": "rol", "horror": "terror"}
CATALOGO = {"accion", "fantasia", "ciencia-ficcion", "rol", "terror"}

crudo = extraer(csv_demo, sep=";")
ver("1. EXTRAÍDO (tal cual venía)", crudo)

limpio, informe = transformar(crudo, MAPA, CATALOGO)
ver("2. TRANSFORMADO", limpio)
ver("Informe de transformación", dict(informe))

validar_etl(limpio, CATALOGO)
print("3. ✓ VALIDADO: todas las invariantes se cumplen")

if HAY_MONGO:
    etl_col = db["etl_obras"]
    etl_col.delete_many({})
    cargados = cargar(limpio, etl_col)
    ver("4. CARGADO", {"documentos": cargados, "en la base": etl_col.count_documents({})})

    # LA PRUEBA DE IDEMPOTENCIA: correrlo otra vez no debe cambiar el total
    cargar(limpio, etl_col)
    ver("¿Idempotente? (el total NO cambió)", etl_col.count_documents({}))
else:
    cargados = len(limpio)
    ver("4. Sin Mongo: filas listas para cargar", cargados)

# LA ECUACIÓN DE CONSERVACIÓN: el número que defiendes en la presentación
rechazados = (informe["rechazo_sin_productor"] + informe["rechazo_duplicado"]
              + informe["rechazo_sin_etiquetas"])
ver("Auditoría de la carga",
    {"leídos": informe["leidos"], "cargados": cargados, "rechazados": rechazados,
     "¿cuadra?": informe["leidos"] == cargados + rechazados})
print("""
  Ese "¿cuadra? True" es el resultado más importante del notebook. Significa que
  cada registro de entrada está CONTABILIZADO: o entró, o fue rechazado por una
  regla con nombre. Un ETL donde la ecuación no cuadra está perdiendo datos en
  algún filtro que nadie recuerda haber escrito.

  Y fíjate en 'ejemplos_desconocidas': 'cosa-inventada' no se cayó en silencio,
  quedó registrada. Eso es lo que te permite decidir si hay que ampliar el
  catálogo o si es basura.""")


▸ 1. EXTRAÍDO (tal cual venía)


,titulo,productor,generos
0,Alfa,Acme,"Action, Fantasy"
1,Beta,acme,sci-fi
2,Gamma,Zeta Co,"Role-Playing, Horror"
3,Gamma,Zeta Co,"Role-Playing, Horror"
4,Delta,NaN,Action
5,Epsilon,Omega,cosa-inventada
6,Zeta,Omega,"Action, cosa-inventada"



▸ 2. TRANSFORMADO


,titulo,productor,generos,tags
0,Alfa,acme,"Action, Fantasy","[accion, fantasia]"
1,Beta,acme,sci-fi,[ciencia-ficcion]
2,Gamma,zeta co,"Role-Playing, Horror","[rol, terror]"
3,Zeta,omega,"Action, cosa-inventada",[accion]



▸ Informe de transformación
{'leidos': 7, 'rechazo_sin_productor': 1, 'rechazo_duplicado': 1, 'etiquetas_desconocidas': 1, 'ejemplos_desconocidas': ['cosa-inventada'], 'rechazo_sin_etiquetas': 1}
3. ✓ VALIDADO: todas las invariantes se cumplen

▸ 4. Sin Mongo: filas listas para cargar
4

▸ Auditoría de la carga
{'leídos': 7, 'cargados': 4, 'rechazados': 3, '¿cuadra?': True}

  Ese "¿cuadra? True" es el resultado más importante del notebook. Significa que
  cada registro de entrada está CONTABILIZADO: o entró, o fue rechazado por una
  regla con nombre. Un ETL donde la ecuación no cuadra está perdiendo datos en
  algún filtro que nadie recuerda haber escrito.

  Y fíjate en 'ejemplos_desconocidas': 'cosa-inventada' no se cayó en silencio,
  quedó registrada. Eso es lo que te permite decidir si hay que ampliar el
  catálogo o si es basura.


---

<a id="sec-31"></a>
## 31. 🔧 Pruébalo tú mismo
<sub>[↑ índice](#indice)</sub>

Modifica y vuelve a correr. Cada una toma un par de minutos y deja una conclusión.

1. **Rompe la normalización.** En `transformar`, quita `.str.lower()`. ¿Cuántos
   productores distintos aparecen ahora? Esa diferencia es exactamente el número de
   stands fantasma que tendrías.

2. **Rompe la idempotencia.** Cambia `cargar` para que use `insert_one` en vez de
   `replace_one(..., upsert=True)`. Corre el ETL dos veces y mira
   `count_documents({})`. ¿Cuántos documentos sobran?

3. **Quiebra la ecuación de conservación.** Añade en `transformar` un filtro extra
   (`d = d[d["titulo"].str.len() > 4]`) **sin** contarlo en el informe. Vuelve a correr
   la auditoría. ¿Qué pasa con `¿cuadra?`, y en cuánto se desvía?

4. **Mide el efecto real del IDF.** Amplía `docs` en §21 a 12 obras con etiquetas
   variadas (algunas rarísimas). Compara los top-2 de Jaccard simple y ponderado. ¿En
   cuántas obras cambia el ranking? Si no cambia en ninguna, el IDF no te sirve para
   ese catálogo — y eso también es un hallazgo.

5. **Rompe el índice compuesto.** En §28 crea `(precio, stand_id)` en vez de
   `(stand_id, precio)` y vuelve a pedir el plan de la consulta que filtra por stand y
   ordena por precio. ¿Qué cambia en `totalDocsExamined`?

6. **Haz que el `merge` multiplique.** En §15 construye una tabla derecha con la clave
   repetida 3 veces y haz el merge **sin** `validate=`. Compara `len()` antes y después.
   Ahora añade `validate="one_to_one"` y lee el mensaje de error.

---

<a id="sec-32"></a>
## 32. Cheat Sheet
<sub>[↑ índice](#indice)</sub>

### Lectura y perfilado
| Qué quiero | Cómo |
|---|---|
| CSV europeo con nulos textuales | `pd.read_csv(f, sep=";", decimal=",", na_values=["N/A","-"])` |
| Mirar antes de cargar todo | `pd.read_csv(f, nrows=1000)` |
| Archivo que no cabe | `for t in pd.read_csv(f, chunksize=100_000):` |
| Conservar los tipos | `.to_parquet()` / `pd.read_parquet()` |
| Perfil completo | `perfil(df)` (§3) |
| Nulos por columna | `df.isna().mean().round(2)` |

### Limpieza
| Qué quiero | Cómo |
|---|---|
| Texto a número, sin reventar | `pd.to_numeric(s, errors="coerce")` |
| Entero **con** nulos | `.astype("Int64")` (mayúscula) |
| Normalizar etiquetas | `normalizar(s)` (§9) |
| Quitar tildes | `unicodedata.normalize("NFKD", x)` + `combining` |
| Duplicados, conservando el último | `drop_duplicates(subset=[...], keep="last")` |

### Transformación
| Qué quiero | Cómo |
|---|---|
| Texto con comas → una fila por valor | `.str.split(",").explode()` |
| Listas → columnas 0/1 | `.str.get_dummies(sep=", ")` |
| Texto `"['a','b']"` → lista real | `.map(ast.literal_eval)` |
| Resumen por grupo | `.groupby(k).agg(n=("x","size"), t=("y","sum"))` |
| El total del grupo en cada fila | `.groupby(k)["x"].transform("sum")` |
| if/else vectorizado | `np.where(cond, a, b)` / `np.select([...],[...])` |
| Merge seguro | `merge(..., validate="one_to_one", indicator=True)` |

### Similitud
| Qué | Fórmula | Código |
|---|---|---|
| Jaccard | `|A∩B|/|A∪B|` | `len(a&b)/len(a|b)` |
| IDF | `log(N/df(t))` | `math.log(N/df[t])` |
| Jaccard ponderado | `Σidf(∩)/Σidf(∪)` | §21 |
| Coseno (todos los pares) | `Mn @ Mn.T` | `coseno_matricial` (§21) |
| Top-k sin uno mismo | — | `puntajes[i] = -np.inf` |

### MongoDB
| Qué quiero | Cómo |
|---|---|
| Confirmar la conexión | `cli.admin.command("ping")` |
| DataFrame → documentos | `a_documentos(df)` (§25) |
| Modificar un campo | `update_one(f, {"$set": {...}})` |
| Carga idempotente | `replace_one(clave, doc, upsert=True)` |
| Añadir sin duplicar en array | `{"$addToSet": {"tags": t}}` |
| El array contiene | `{"tags": "accion"}` |
| Contiene todos | `{"tags": {"$all": [...]}}` |
| explode + value_counts | `$unwind` → `$group` → `$sort` |
| Join | `$lookup` → `$unwind` |
| Ver si usó índice | `.explain()` → `IXSCAN` vs `COLLSCAN` |
| Índice único | `create_index([(k,1)], unique=True)` |

### Números duros para recordar
| Dato | Valor |
|---|---|
| Tamaño máximo de un documento BSON | **16 MB** |
| `idf` de una etiqueta presente en todos | **0** |
| Factor vectorizado vs `apply(axis=1)` | **decenas a cientos de ×** |
| Réplica típica en HDFS/Mongo | **×3** |
| Orden de un índice compuesto | **E → S → R** (equality, sort, range) |

---

<a id="sec-33"></a>
## 33. Conexiones con el Mundo Real
<sub>[↑ índice](#indice)</sub>

**Lo que acabas de construir no es un ejercicio escolar.** Es, en miniatura, el trabajo
de un *data engineer*:

- **El informe de rechazos** es lo que en producción se llama *data quality report*, y
  es lo primero que un equipo revisa cuando un dashboard muestra una cifra rara.
- **La ecuación `leídos = cargados + rechazados`** es *reconciliación*: en un banco, un
  pipeline que no reconcilia es un pipeline que no se puede auditar.
- **La tabla de equivalencias como CSV editable** es un *data contract* ligero: el
  vocabulario vive fuera del código y lo mantiene quien conoce el dominio, no quien
  programa.
- **`replace_one(..., upsert=True)`** es la razón por la que un ETL nocturno puede
  reintentarse a las 3 AM sin que nadie revise si duplicó. Airflow, Dagster y Spark
  Structured Streaming construyen toda su semántica de reintentos sobre esa propiedad.
- **El `explain()`** es la misma herramienta con la que un DBA explica por qué una
  consulta que tardaba 40 ms ahora tarda 8 segundos: alguien agregó una columna al
  `ORDER BY` y el índice compuesto dejó de servir.
- **Jaccard + IDF** es la base de los sistemas de recomendación por contenido, y el
  escalón previo a MinHash/LSH, que es cómo se hace lo mismo cuando hay millones de
  ítems y comparar todos los pares es imposible.

En los tres ramos donde esto reaparece:

- **Big Data:** esta es la Entrega 2 completa, con ETL + almacenamiento + consultas.
- **IA (NB-09 a NB-11):** un modelo entrenado sobre datos con fuga o con etiquetas sin
  normalizar da métricas excelentes que no se reproducen. El 80 % del trabajo de ML es
  esta parte.
- **Sistemas Distribuidos:** §29 es replicación líder-seguidor y particionamiento, con
  el mismo trade-off de consistencia (`w=1` vs `w="majority"`) que verás formalizado
  como CAP/PACELC.

---

<a id="sec-34"></a>
## 34. Ejercicios
<sub>[↑ índice](#indice)</sub>

### Nivel 1 — Mecánica (deberías poder hacerlos sin mirar atrás)

1. Lee un CSV con `;`, coma decimal y `"N/A"` como nulo, y verifica con `.dtypes` que
   la columna de montos quedó `float64` y no `object`.
2. Dada una columna `"Action, Fantasy"`, produce la tabla larga `(id, tag)` con las
   etiquetas en minúsculas y sin espacios.
3. Cuenta cuántas filas tienen etiqueta vacía **después** de normalizar, y explica en
   una línea por qué el conteo puede subir respecto de antes de normalizar.
4. Convierte un DataFrame con una columna `datetime` **que tenga nulos** a documentos
   válidos para Mongo, y demuestra con `json.dumps` que no quedó ningún `NaT`.

### Nivel 2 — Aplicación (requieren decidir algo)

5. Escribe `perfilar_fuente(df)` que devuelva, en una sola fila: filas, % de nulos de
   la columna de productor, nº de etiquetas distintas, mediana de etiquetas por ítem y
   share del top-3 de productores. Córrelo sobre dos fuentes y **decide con la tabla**
   cuál sirve mejor como fuente principal. Justifica con los números, no con la
   intuición.
6. Construye la tabla de equivalencias de etiquetas de **una** de tus fuentes como CSV.
   Mide la cobertura (`% de etiquetas mapeadas`) e itera hasta pasar el 85 %. Reporta
   qué quedó fuera y por qué decidiste dejarlo fuera.
7. Implementa `top_k` con Jaccard ponderado por IDF sobre 200 ítems reales y compara
   con Jaccard simple. Reporta: en cuántos ítems cambia el top-1, y el solapamiento
   medio del top-5. Si el efecto es menor al 5 %, explica por qué (mira el rango de
   los `idf`).
8. Añade al ETL una regla de negocio nueva —la que tú quieras— que **rechace** filas.
   Asegúrate de que la ecuación de conservación siga cuadrando y de que el informe
   nombre la regla.

### Nivel 3 — Nivel proyecto (son decisiones de diseño, no de código)

9. **El dilema del 41 % de nulos.** `Studios` de anime tiene 41 % de nulos. Escribe
   media página defendiendo **una** de las dos opciones (descartar vs stand "Estudio
   desconocido"), con: qué se pierde en cada caso, qué pasa con el recomendador si un
   solo stand concentra 11.614 obras, y cómo lo verificarías con una medición.
10. **El filtro de `popular_shelves`.** Las estanterías de Goodreads mezclan géneros con
    ruido (`to-read`, `favorites`). Diseña el criterio de filtrado, **impleméntalo**, y
    mide cuántas etiquetas sobreviven con tres umbrales distintos. Elige uno y defiende
    la elección con la medición.
11. **Mongo vs PostgreSQL, por escrito.** Responde la pregunta de la defensa en un
    párrafo que incluya, obligatoriamente: un beneficio concreto de tu caso, el nombre
    técnico del mecanismo que lo da, y **una cosa que pierdes**. Sin la pérdida, la
    respuesta no es defendible.
12. **El índice que de verdad necesitas.** Lista las 3 consultas más frecuentes de tu
    aplicación. Para cada una, decide el índice (campo, orden, compuesto o no) y
    justifica con la regla ESR. Después mídelo con `explain()` y compara
    `totalDocsExamined` antes y después. Si no baja, el índice está mal elegido.

---

<a id="sec-35"></a>
## 35. Glosario
<sub>[↑ índice](#indice)</sub>

| Término | Definición precisa |
|---|---|
| **Vectorización** | Aplicar una operación a todo un array en código compilado, sin bucle del intérprete. |
| **Broadcasting** | Regla de NumPy que estira virtualmente dimensiones de tamaño 1 para operar formas distintas. |
| **Máscara booleana** | Series/array de `True`/`False` del mismo largo, usada para indexar. |
| **Tidy data** | Una observación por fila, una variable por columna. La forma "larga". |
| **split–apply–combine** | El patrón de `groupby`: partir por clave, aplicar, recombinar. |
| **`document frequency` (df)** | Nº de documentos que contienen una etiqueta. |
| **IDF** | `log(N/df)`. Peso informativo: alto para lo raro, 0 para lo omnipresente. |
| **Jaccard** | `|A∩B|/|A∪B|`. Similitud entre conjuntos. |
| **Ochiai** | El coseno aplicado a vectores binarios: `|A∩B|/√(|A||B|)`. |
| **Idempotencia** | Ejecutar la operación N veces deja el mismo estado que ejecutarla una. |
| **Upsert** | Actualizar si existe, insertar si no. El mecanismo que da idempotencia. |
| **Fuga de datos** *(leakage)* | Información del futuro o del conjunto de prueba filtrada al entrenamiento. |
| **BSON** | Formato binario de MongoDB. Tipado, con límite de 16 MB por documento. |
| **Colección / documento** | Equivalentes documentales de tabla y fila. |
| **`$unwind`** | Etapa que genera un documento por elemento de un array. El `explode` de Mongo. |
| **COLLSCAN / IXSCAN** | Recorrido completo de la colección vs. uso de índice. |
| **Índice multikey** | Índice sobre un campo array: indexa cada elemento por separado. |
| **Regla ESR** | Orden de campos en un índice compuesto: Equality → Sort → Range. |
| **Write concern** | Cuántos nodos deben confirmar una escritura (`w=1`, `w="majority"`). |
| **Shard key** | Campo que decide en qué partición vive cada documento. |
| **Hot spot** | Partición que recibe desproporcionadamente la carga; síntoma: **subutilización del clúster**. |
| **Rack awareness** | Política de colocación de réplicas en racks distintos para sobrevivir a la caída de uno. |

---

<a id="sec-36"></a>
## 36. Conclusión y verificación
<sub>[↑ índice](#indice)</sub>

### Lo que este notebook deja instalado

Cuatro cosas, en orden de importancia:

1. **Perfilar antes de decidir.** Las decisiones de limpieza salen de números medidos
   (`% nulos`, `mediana de etiquetas`, `share del top-3`), no de intuiciones.
2. **El error se declara o no existe.** `errors="coerce"` + informe + `assert` +
   ecuación de conservación. Un pipeline callado es un pipeline que está perdiendo
   datos.
3. **Idempotencia por diseño.** `replace_one(clave, doc, upsert=True)`. Es lo que
   separa un ETL que puedes reintentar de uno que te obliga a borrar la base.
4. **Vectorizar, y medirlo.** La diferencia con `apply(axis=1)` es de órdenes de
   magnitud, y `memory_usage(deep=True)` + `explain()` son las dos herramientas que
   convierten "parece lento" en un número.

### Verificación de comprensión

Respóndelas **con tus palabras**, sin volver a subir. Si alguna te cuesta, esa es la
sección que hay que releer.

1. Tienes dos columnas de etiquetas, una con mediana 1 etiqueta por ítem y otra con
   mediana 4. ¿Por qué Jaccard es casi inútil en la primera? ¿Qué medirías antes de
   elegir la medida de similitud?
2. Un compañero corrió el ETL dos veces y la colección tiene el doble de documentos.
   ¿Qué línea exacta hay que cambiar, y por qué eso lo arregla?
3. `leídos = 16.598`, `cargados = 14.203`, `rechazados = 1.900`. ¿Qué está mal, y qué
   harías para encontrarlo?
4. Explica, sin usar la palabra "rápido", qué hace un índice y qué cuesta tenerlo.
5. Tu `merge` de stands con etiquetas devolvió 23.000 filas donde esperabas 16.598.
   ¿Cuál es la causa más probable y con qué parámetro lo habrías detectado el día uno?
6. ¿Por qué `w="majority"` es más lento que `w=1`, y en qué caso concreto de tu
   proyecto elegirías cada uno?
7. Una fecha ambigua (`01/11/2026`) entró mal en tu pipeline. ¿Qué parámetro faltaba, y
   por qué no se detectó con un `assert`?

### Navegación

- **Antes:** NB-01 (jerarquía de memoria, NumPy) · `guia_pandas` (referencia rápida)
- **Este:** tutorial completo pandas + NumPy + MongoDB
- **Después:** NB-02 (el caso GAMECON completo, con decisiones de diseño) ·
  NB-03 (vectorización en profundidad) · NB-04 (similitud y recorridos) ·
  NB-22 (este ETL convertido en DAG de Airflow)

> **Lo que sigue, concretamente:** no vuelvas a leer este notebook. Abre tus datos
> reales, corre `perfil()` sobre las tres fuentes, y responde con números las dos
> decisiones que están pendientes: qué hacer con el 41 % de nulos de `Studios`, y qué
> criterio usar para filtrar `popular_shelves`. Las dos son decisiones del grupo, y las
> dos se defienden con una medición que ya sabes hacer.